# CM03 - Laplacian-smoothed grain boundary interfaces

Extract internal grain boundary surfaces directly from the voxel labels, smooth them with Laplacian relaxation, and display only those surfaces. No tetrahedra are generated. RVE exterior caps are omitted from this view.

Surface nodes smooth along surface adjacency; junction-line nodes use junction neighbors. Junction endpoints, branches and four-or-more-grain points stay frozen. RVE-plane coordinates remain fixed. This preview has triangle-collapse safeguards, but no volume-element quality constraints; its surface geometry can differ from CM01. It does not certify absence of self-intersections.

Run from inside this checkout with NumPy and PyVista installed in the notebook kernel.

## Control guide

These notes describe the existing controls. After changing an upstream setting, rerun its cell and all dependent cells below it. Assignments such as `CAP_MESH_SIZE = GMSH_SIZE` take effect when that cell runs; they are not dynamic links. Physical lengths use the units of `SPACING`. Increasing a limit helps only if that limit is actually preventing progress.

### Input and voxel cleaning

| Control | Meaning; effect of increasing or decreasing |
|---|---|
| `SAMPLE`, `ROOT` | Input data and repository paths. Changing the sample changes the structure. |
| `GRAIN_ID_SEED` | Reproducible random ID/colour permutation. Different integers give different permutations; larger is not necessarily more contrasting. Geometry is unchanged. |
| `CLEAN_SPIKES` | `True` enables UPXO spike removal before refinement; `False` bypasses it. |
| `SPIKE_MAX_PASSES` | Higher allows more cleaning passes; lower stops sooner. No-change and repeated-configuration stops still apply. More passes cannot override protected last voxels. |
| `CLEAN_PINCHES` | Enables targeted voxel contact cleaning; `False` bypasses it. |
| `PINCH_MAX_PASSES` | Higher allows more search passes; lower limits work. Other budgets and acceptance rules still apply. |
| `PINCH_MAX_CHANGES` | Higher allows more accepted voxel reassignments in total; lower is more conservative. |
| `PINCH_MAX_VOLUME_FRACTION` | Higher permits larger per-grain net voxel-count changes; lower preserves counts more tightly. Positive values allow at least one voxel; zero allows none. This is not a displacement limit or total-edit fraction. |
| `PINCH_PROTECT_GRAINS_UP_TO` | Higher protects more small grains from donating voxels; lower permits cleaning smaller grains. Counts are measured after spike cleaning, before upscaling. |
| `UPSCALE_FACTOR` | Integer ≥1: 1 leaves resolution unchanged; higher subdivides along all axes. Memory grows approximately with factor³. Refinement alone preserves the original geometry and contacts; it adds no physical detail. |
| `SPACING` | Original voxel dimensions x/y/z. Increasing a component enlarges that physical dimension; decreasing shrinks it. Use measured spacing, not this setting as a repair control. |

`effective_spacing` is derived from `SPACING / UPSCALE_FACTOR`. Settings using `min(SPACING)` remain tied to original physical voxel size. The pinch detector searches local 2×2×2 face-disconnected arrangements. Its budgets cannot extend detection to every thin neck or later smoothing-induced defect. Accepted edits preserve global six-connected component counts and obey volume/protection rules.

### Surface and junction smoothing

| Control | Increase / decrease effect |
|---|---|
| `LAPLACE_ITERATIONS` | More surface-smoothing steps can reduce staircasing but increase shrinkage, distortion and runtime; fewer retain more voxel detail. Zero disables these steps. |
| `LAPLACE_RELAXATION` | Larger values give stronger averaging steps; smaller are gentler. Use the supported range (0,1]. Local safeguards can suppress movement. |
| `JUNCTION_ITERATIONS` | More steps smooth eligible junction lines further; fewer retain more original shape. Zero disables line smoothing. |
| `JUNCTION_RELAXATION` | Larger values give stronger line-smoothing steps; smaller are gentler. Range (0,1]. |
| `JUNCTION_MAX_DISPLACEMENT` | Larger physical bound permits more total junction movement from extracted geometry; smaller preserves shape more tightly. Extra iterations may do little once constrained. |

Junction endpoints, branches and four-or-more-grain nodes remain frozen, and RVE-plane coordinates stay fixed. Stronger smoothing does not remove these topology constraints and can bring neighbouring sheets closer. Local collapse/reversal safeguards do not certify freedom from global intersections.

### Pre-Gmsh point-contact separation

| Control | Increase / decrease or switch effect |
|---|---|
| `SEPARATE_PINCHES` | Enables splitting disconnected triangle fans at a shared vertex; `False` bypasses it. |
| `PINCH_PRESS_RADIUS` | Larger physical radius affects a broader neighbourhood; smaller localizes pressing near the split tip. |
| `PINCH_PRESS_MAX_DISPLACEMENT` | Larger allows more movement; smaller better preserves shape but may prevent separation. |
| `PINCH_PRESS_ITERATIONS` | More steps permit further adjustment; fewer reduce work and deformation. Constraints still apply. |
| `PINCH_PRESS_RELAXATION` | Larger gives stronger steps; smaller gives gentler adjustment. |
| `PINCH_MIN_OPENING` | Larger demands a wider minimum separation of duplicated tips and can cause rollback if unattainable; smaller accepts narrower separation. It is not a global clearance guarantee. |

Length controls use physical units. This operation handles disconnected point fans: it does **not** cut shared edges or connected multi-grain junctions. Increasing its controls cannot remove contacts outside that scope. Other frozen junction points and RVE coordinates remain constrained.

### Gmsh and RVE caps

| Control | Increase / decrease effect |
|---|---|
| `GMSH_SIZE` | Smaller requests finer surface triangles and greater cost; larger requests coarser triangles. Retained curves and chart seams constrain size. Refinement does not itself remove contacts or overlaps. |
| `GMSH_ALGORITHM` | Categorical: 1 MeshAdapt, 2 Automatic, 5 Delaunay, 6 Frontal-Delaunay. Higher numbers are not inherently better. |
| `CAP_MESH_SIZE` | Smaller requests finer new cap triangles; larger requests coarser ones. |
| `RVE_EDGE_SIZE` | Smaller subdivides new box-edge segments more finely; larger permits longer segments. Existing interface traces remain constrained. |
| `FINAL_GMSH_SIZE` | Same size tradeoff, applied to joint remeshing of internal interfaces and caps. |
| `FINAL_GMSH_ALGORITHM` | Same categorical choices as `GMSH_ALGORITHM`. |

The reported “internal triangle is coplanar with the cap” error is a geometry/classification check. Changing cap size alone does not address that condition. Its investigation and repair are addressed by the CM03 upstream geometry guard.

### Validation, later repairs and tetrahedra

| Control | Increase / decrease or switch effect |
|---|---|
| `MIN_SURFACE_TRIANGLE_QUALITY` | Higher flags more poor triangles; lower accepts more. This is a checking threshold, not an improvement operation. |
| `STOP_ON_VALIDATION_ERRORS` | `True` raises after saving the report; `False` permits later cells but does not make failed geometry valid. |
| `REPAIR_CONTACTS` | Enables the later contact repair; `False` bypasses it. |
| `REPAIR_SEPARATION` | Larger physical movement limit permits stronger separation; smaller is more conservative. Only supported ambiguous edges/fans are handled; genuine multi-grain junction edges remain shared. |
| `REPAIR_INTERFACE_CROSSINGS` | Enables the supported local crossing repair; `False` bypasses it. |
| `CROSSING_MAX_DISPLACEMENT` | Larger permits larger physical corrections; smaller can reject required movement. |
| `CROSSING_NEIGHBOURHOOD_RADIUS` | Larger uses a broader region to estimate separation directions; smaller uses more local normals. It is not a general collision-search distance. |
| `CROSSING_OPENING` | Larger dimensionless value requests wider wedge separation relative to distance from the junction line; smaller is gentler. Excessive opening can violate movement or inversion checks. |
| `RUN_TET_VERIFICATION` | Enables actual tetrahedral generation and checks. Disabling it does not establish readiness. |
| `TET_MESH_SIZE` | Smaller requests finer volume elements at greater cost; larger requests coarser ones, subject to surface constraints. Neither direction guarantees better minimum quality. |
| `TET_OPTIMIZE_NETGEN` | Enables optimization, retaining the baseline for a volume when its minimum quality worsens. |
| `MIN_TET_QUALITY` | Higher makes acceptance stricter; lower admits worse tetrahedra. It does not itself improve elements. |

Crossing repair supports particular interface wedges beside shared junction lines, not arbitrary overlaps or coplanar triangles. Revalidation is required after repairs.

### Display and output controls

| Control | Effect |
|---|---|
| `GRAIN_ID` | `None` shows all interfaces; a shuffled ID selects that grain's interfaces. |
| `AUDIT_ORIGINAL_GRAIN_ID` | Original unshuffled ID to trace; `None` selects a smallest input grain. |
| `OPACITY`, `CAP_OPACITY` | Higher is more opaque; lower more transparent (0–1). Transparency can complicate interpretation. |
| `SHOW_EDGES`, `SHOW_JUNCTION_LINES`, `SHOW_AUDIT_GRAIN`, `GMSH_SHOW_CURVES`, `SHOW_INTERNAL_INTERFACES`, `CAP_SHOW_EDGES`, `FINAL_SHOW_INTERNAL`, `PLOT_VALIDATION_ISSUES`, `SHOW_REPAIRED_SURFACES`, `SHOW_TET_CRINKLE_CLIP` | Display switches only: `True` shows the feature, `False` hides it. Gmsh curves can include computational seams. Crinkle clipping displays whole intersected tetrahedra. |
| `GMSH_VERBOSE` | Enables detailed Gmsh messages; no intended geometry change. |
| Names ending in `_OUTPUT` or `_AUDIT` | Saved-file locations, not numerical controls. |

**CM03 repair scope:** contact/overlap diagnosis, more rigorous voxel cleaning and the RVE coplanarity error are authorized for CM03. The CM03 stages below implement bounded repairs and report unresolved defects. Images alone cannot establish every shared-node contact or triangle intersection.


In [1]:
from pathlib import Path
import sys
import importlib
import numpy as np

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/upxo/meshing/gbconformant').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Start Jupyter inside the upxo_library checkout.')
sys.path.insert(0, str(ROOT / 'src'))
from upxo.meshing.gbconformant.d3v2p0.paths import data_directory
DATA_ROOT = data_directory()
RUN_DIR = DATA_ROOT / 'runs' / 'CM03'
RUN_DIR.mkdir(parents=True, exist_ok=True)
# Refresh shared helpers as well as entry points in an existing kernel.
for _name in ('facet_angles', 'surface_intersections', 'surface_charts', 'discrete_topology'):
    importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.' + _name))
interface_module = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.interfaces'))
smooth_interfaces = interface_module.smooth_interfaces
SAMPLE = DATA_ROOT / 'samples/sblk_lgi.npy'
labels_original = np.load(SAMPLE, allow_pickle=False)

GRAIN_ID_SEED = 42
original_grain_ids = np.unique(labels_original)
shuffled_grain_ids = np.random.default_rng(GRAIN_ID_SEED).permutation(original_grain_ids)
grain_id_map = dict(zip(original_grain_ids.tolist(), shuffled_grain_ids.tolist()))
labels = shuffled_grain_ids[np.searchsorted(original_grain_ids, labels_original)]
print(f'{labels.shape}: {len(original_grain_ids)} grains')

(50, 50, 50): 373 grains


## Iterative UPXO spike removal before upscaling
Uses the existing `StructureCleaner3D._fast_clean_spikes` routine from `upxo.pxtal.twinned_simple_3d.cleaning_3d`. A spike has at most one face-connected neighbor with the same grain ID. Each pass reassigns spikes to neighboring grains while protecting the last voxel of every grain.

Set `CLEAN_SPIKES = False` to bypass cleaning, or adjust `SPIKE_MAX_PASSES`. Cleaning stops when a pass changes no voxels, the pass limit is reached, or a label configuration repeats. Remaining spike candidates can include protected singleton grains. Cleaning changes grain shapes and voxel counts; the original `labels` and input file remain intact. The optional pinch-cleaning step receives `labels_cleaned` before upscaling.


In [2]:
CLEAN_SPIKES = True
SPIKE_MAX_PASSES = 100

# Always start from loaded/shuffled labels, not the result of a previous run.
labels_cleaned = labels.copy()
spike_history = []
spike_stop_reason = 'disabled'
if CLEAN_SPIKES:
    import hashlib
    from upxo.pxtal.twinned_simple_3d.cleaning_3d import StructureCleaner3D

    if isinstance(SPIKE_MAX_PASSES, bool) or not isinstance(SPIKE_MAX_PASSES, (int, np.integer)) or SPIKE_MAX_PASSES < 1:
        raise ValueError('SPIKE_MAX_PASSES must be a positive integer')
    if np.any(labels_cleaned < 0):
        raise ValueError('The UPXO spike cleaner requires nonnegative labels.')
    spike_cleaner = StructureCleaner3D(upscale_fallback=False, do_lobe_split=False)
    grain_ids_before_cleaning = np.unique(labels_cleaned)
    seen_states = {hashlib.sha256(labels_cleaned.tobytes()).digest()}
    spike_stop_reason = 'maximum passes reached'
    for pass_number in range(1, SPIKE_MAX_PASSES + 1):
        labels_cleaned, changed = spike_cleaner._fast_clean_spikes(labels_cleaned)
        spike_history.append({'pass': pass_number, 'changed_voxels': int(changed)})
        print(f'Spike pass {pass_number}: {changed} voxels reassigned')
        if changed == 0:
            spike_stop_reason = 'no further changes'
            break
        state = hashlib.sha256(labels_cleaned.tobytes()).digest()
        if state in seen_states:
            spike_stop_reason = 'repeated label configuration'
            break
        seen_states.add(state)
    assert np.array_equal(np.unique(labels_cleaned), grain_ids_before_cleaning), 'Cleaning lost a grain ID'
    remaining_candidates = np.count_nonzero(
        (labels_cleaned > 0) & (spike_cleaner._count_face_same_grain(labels_cleaned)
                               <= spike_cleaner._SPIKE_SAME_COUNT_THRESHOLD))
    print(f'Remaining spike candidates (including protected last voxels): {remaining_candidates}')
print('Spike cleaning stopped:', spike_stop_reason)
print('Net changed voxels:', np.count_nonzero(labels_cleaned != labels))
print('Grains retained:', len(np.unique(labels_cleaned)))


Spike pass 1: 950 voxels reassigned
Spike pass 2: 212 voxels reassigned
Spike pass 3: 37 voxels reassigned
Spike pass 4: 13 voxels reassigned
Spike pass 5: 7 voxels reassigned
Spike pass 6: 3 voxels reassigned
Spike pass 7: 2 voxels reassigned
Spike pass 8: 2 voxels reassigned
Remaining spike candidates (including protected last voxels): 195
Spike cleaning stopped: repeated label configuration
Net changed voxels: 1183
Grains retained: 373


## Optional targeted voxel pinch cleaning
Run after spike removal and before upscaling. `CLEAN_PINCHES` turns the step on or off. The detector searches 2 x 2 x 2 blocks for same-grain voxels connected only through edges/corners locally. It does not apply a majority filter. These are candidates, not proof that the contact is physically wrong; thin necks and pinches introduced later by smoothing are outside this detector's scope.

Each accepted one-voxel edit must remove candidates without introducing new ones, retain every grain ID, and preserve each grain's global face-connected component count. This does not guarantee preservation of tunnels, cavities, or all grain adjacency relationships. Small grains cannot donate voxels. Per-grain net volume change is limited to the greater of one voxel and the specified fraction of its initial count; a zero fraction forbids volume changes.

Controls cap passes, accepted edits, volume change and small-grain protection. Unresolved candidates remain reported. The input `labels_cleaned` remains unchanged; `labels_pinch_cleaned` feeds upscaling. The JSON audit records every voxel reassignment for inspection or replay. Compare on/off results with identical smoothing and Gmsh settings.


In [3]:
CLEAN_PINCHES = True  # False bypasses this step completely.
PINCH_MAX_PASSES = 100
PINCH_MAX_CHANGES = 200
PINCH_MAX_VOLUME_FRACTION = 0.03
PINCH_PROTECT_GRAINS_UP_TO = 8  # Original voxel count, after spike cleaning.

import json
_pinch_module = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.pinch_cleaning'))
labels_pinch_cleaned, pinch_report, pinch_changes = _pinch_module.clean_pinch_contacts(
    labels_cleaned, enabled=CLEAN_PINCHES,
    max_passes=PINCH_MAX_PASSES, max_changes=PINCH_MAX_CHANGES,
    max_volume_fraction=PINCH_MAX_VOLUME_FRACTION,
    protect_grains_up_to=PINCH_PROTECT_GRAINS_UP_TO)
print(json.dumps(pinch_report, indent=2))
PINCH_AUDIT = RUN_DIR / 'CM03_pinch_cleaning.json'
PINCH_AUDIT.parent.mkdir(parents=True, exist_ok=True)
PINCH_AUDIT.write_text(json.dumps({'report': pinch_report, 'changes': pinch_changes}, indent=2))
print('Pinch-cleaning audit:', PINCH_AUDIT)


{
  "enabled": true,
  "initial_candidates": 482,
  "remaining_candidates": 261,
  "accepted_changes": 200,
  "net_changed_voxels": 200,
  "grains_retained": 373,
  "face_component_counts_preserved": true,
  "stop_reason": "maximum changes reached",
  "passes": [
    {
      "pass": 1,
      "accepted": 200,
      "remaining_candidates": 261
    }
  ],
  "grain_voxel_changes": {
    "3": 2,
    "4": 5,
    "6": -1,
    "15": -1,
    "17": 1,
    "23": -2,
    "24": 1,
    "34": -2,
    "35": -2,
    "45": -4,
    "46": 2,
    "51": 2,
    "52": 4,
    "53": 1,
    "72": 1,
    "78": -3,
    "81": -1,
    "86": 3,
    "91": -1,
    "106": -1,
    "110": 1,
    "112": -3,
    "114": -2,
    "116": 1,
    "122": 2,
    "124": 2,
    "129": -2,
    "131": 2,
    "138": -1,
    "139": 1,
    "143": -1,
    "184": 1,
    "185": 1,
    "190": -4,
    "198": 4,
    "200": 1,
    "204": -3,
    "205": 1,
    "207": -1,
    "208": 1,
    "209": 1,
    "211": -1,
    "213": -2,
    "221": -3,
   

### Rigorous voxel boundary topology cleaning

This checks whether every grain has a single, non-branching boundary link at each lattice vertex, including RVE faces, edges and corners. It detects point contacts, alternating edge contacts and disconnected complementary regions. Single-voxel edits and bounded two-voxel edits reduce the defect count; every grain ID is retained. This is targeted topology cleaning, not a majority filter.

Higher pass/change/pair-trial budgets allow more search. A higher volume fraction permits larger **net** per-grain count changes; zero still permits count-preserving swaps. Raising the small-grain protection threshold protects more donors. `TOPOLOGY_PRESERVE_COMPONENTS=False` permits changes to connected-component counts, including removal of legitimate contacts; set True to restrict them. `REQUIRE_CLEAN_TOPOLOGY` stops on remaining defects. The audit records all edits and unresolved vertices. All switches are user-controlled.

A positive volume fraction allows at least one voxel of net count change per grain; this discrete minimum can exceed that fraction for very small grains. Zero allows only count-preserving swaps. Set `TOPOLOGY_MAX_PAIR_TRIALS=0` to disable two-voxel lookahead.


In [4]:
# Exact cubical boundary-link cleaning, including RVE boundary vertices.
CLEAN_VOXEL_TOPOLOGY = True
TOPOLOGY_MAX_PASSES = 20
TOPOLOGY_MAX_CHANGES = 5000
TOPOLOGY_MAX_VOLUME_FRACTION = 0.20
TOPOLOGY_PROTECT_GRAINS_UP_TO = 0
TOPOLOGY_PRESERVE_COMPONENTS = False
TOPOLOGY_MAX_PAIR_TRIALS = 20000
REQUIRE_CLEAN_TOPOLOGY = True

_topology = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.voxel_topology'))
labels_meshing, topology_report, topology_changes = _topology.clean_voxel_topology(
    labels_pinch_cleaned, enabled=CLEAN_VOXEL_TOPOLOGY,
    max_passes=TOPOLOGY_MAX_PASSES, max_changes=TOPOLOGY_MAX_CHANGES,
    max_volume_fraction=TOPOLOGY_MAX_VOLUME_FRACTION,
    protect_grains_up_to=TOPOLOGY_PROTECT_GRAINS_UP_TO,
    preserve_components=TOPOLOGY_PRESERVE_COMPONENTS,
    max_pair_trials=TOPOLOGY_MAX_PAIR_TRIALS)
TOPOLOGY_AUDIT = RUN_DIR / 'CM03_voxel_topology.json'
TOPOLOGY_AUDIT.parent.mkdir(parents=True, exist_ok=True)
TOPOLOGY_AUDIT.write_text(json.dumps({'report': topology_report, 'changes': topology_changes}, indent=2))
print({k:v for k,v in topology_report.items() if k not in ('unresolved', 'grain_voxel_changes')})
assert np.array_equal(np.unique(labels_meshing), np.unique(labels_original))
if REQUIRE_CLEAN_TOPOLOGY and topology_report['remaining_defects']:
    raise RuntimeError('Unresolved voxel boundary defects: inspect TOPOLOGY_AUDIT before continuing.')


{'enabled': True, 'initial_defects': 668, 'remaining_defects': 0, 'accepted_changes': 490, 'net_changed_voxels': 490, 'all_grain_ids_preserved': True, 'preserve_components': False, 'stop_reason': 'no defects remain', 'passes': [{'pass_number': 1, 'accepted': 464, 'remaining': 17}, {'pass_number': 2, 'accepted': 3, 'remaining': 14}, {'pass_number': 3, 'accepted': 2, 'remaining': 13}, {'pass_number': 4, 'accepted': 1, 'remaining': 12}, {'pass_number': 5, 'accepted': 2, 'remaining': 11}, {'pass_number': 6, 'accepted': 1, 'remaining': 10}, {'pass_number': 7, 'accepted': 2, 'remaining': 9}, {'pass_number': 8, 'accepted': 2, 'remaining': 8}, {'pass_number': 9, 'accepted': 1, 'remaining': 7}, {'pass_number': 10, 'accepted': 2, 'remaining': 6}, {'pass_number': 11, 'accepted': 2, 'remaining': 5}, {'pass_number': 12, 'accepted': 2, 'remaining': 4}, {'pass_number': 13, 'accepted': 1, 'remaining': 3}, {'pass_number': 14, 'accepted': 2, 'remaining': 2}, {'pass_number': 15, 'accepted': 1, 'remaining

## Optional resolution increase before smoothing
Set `UPSCALE_FACTOR` to an integer: `1` leaves the data unchanged, `2` doubles the resolution along each axis, and `3` triples it. Each original voxel is subdivided into identically labelled voxels using nearest-neighbor replication; no new grain IDs are introduced. Spacing decreases by the same factor, preserving physical RVE dimensions and grain shapes before smoothing.

For the current 50 x 50 x 50 sample, factor 2 produces 100 x 100 x 100 voxels; factor 3 produces 150 x 150 x 150. This notebook limits refinement to a maximum factor of 3. Voxel memory grows as factor cubed, and interface triangle count as factor squared. At higher resolution, the same Laplacian iteration count generally smooths over a smaller physical distance; increase `LAPLACE_ITERATIONS` if needed. Always refine from `labels_meshing`, so rerunning this cell does not compound the upscaling.


In [5]:
UPSCALE_FACTOR = 2  # 1: unchanged; 2 or 3: subdivide along all three axes.
if UPSCALE_FACTOR not in (1, 2, 3):
    raise ValueError("CM03 supports upscale factors 1, 2, or 3 (maximum).")
SPACING = (1.0, 1.0, 1.0)  # Original voxel spacing, before refinement.

# Refresh definitions if the notebook kernel predates this module update.
interface_module = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.interfaces'))
smooth_interfaces = interface_module.smooth_interfaces
labels_upscaled, effective_spacing = interface_module.upscale_labels(
    labels_meshing, factor=UPSCALE_FACTOR, spacing=SPACING)
print(f'Voxel shape: {labels_meshing.shape} -> {labels_upscaled.shape}')
print(f'Effective voxel spacing: {effective_spacing}')
print(f'Physical RVE dimensions: {np.array(labels_upscaled.shape) * effective_spacing}')


Voxel shape: (50, 50, 50) -> (100, 100, 100)
Effective voxel spacing: [0.5 0.5 0.5]
Physical RVE dimensions: [50. 50. 50.]


## Extract and smooth interfaces only
Change the iteration count and relaxation to control smoothing. Shared interfaces are extracted once per adjacent grain pair. Surface triangles are used for rendering; there is no volume meshing step.

Junction edges are identified from their incident grain interfaces, avoiding false branches caused by surface triangulation diagonals. Genuine junction points (four or more grains), branches and endpoints remain fixed. Movable degree-two junction nodes have independent controls:

- `JUNCTION_ITERATIONS`: line-smoothing steps; 0 disables line motion.
- `JUNCTION_RELAXATION`: smoothing rate; 0.25 is a damped starting value.
- `JUNCTION_MAX_DISPLACEMENT`: maximum total movement from the extracted coordinates, in physical units. The default uses 0.75 original voxel widths, independently of upscaling.

Surface smoothing and line smoothing advance together until their respective iteration limits. Triangle-collapse checks can restrict individual moves; frozen endpoints and branches can retain sharp turns. Increase surface iterations if surrounding surfaces constrain line motion. Turning-angle diagnostics refer only to movable degree-two nodes (0 degrees is straight).


In [6]:
LAPLACE_ITERATIONS = 50
LAPLACE_RELAXATION = 0.75

# Independent controls for actual junction-line edges.
JUNCTION_ITERATIONS = 10
JUNCTION_RELAXATION = 0.35  # Damped Laplacian step; 0 < value <= 1.
JUNCTION_MAX_DISPLACEMENT = 0.75 * min(SPACING)  # Physical units, from extracted geometry.

# Refresh the function even when only this cell is rerun after a source edit.
interface_module = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.interfaces'))
smooth_interfaces = interface_module.smooth_interfaces
interfaces = smooth_interfaces(labels_upscaled, spacing=effective_spacing,
                               iterations=LAPLACE_ITERATIONS,
                               relaxation=LAPLACE_RELAXATION,
                               junction_iterations=JUNCTION_ITERATIONS,
                               junction_relaxation=JUNCTION_RELAXATION,
                               junction_max_displacement=JUNCTION_MAX_DISPLACEMENT)
assert np.array_equal(interfaces.points[interfaces.node_kind == 3],
                      interfaces.original_points[interfaces.node_kind == 3])
assert np.array_equal(interfaces.points[interfaces.fixed_axes],
                      interfaces.original_points[interfaces.fixed_axes])
print(f'{len(interfaces.points):,} surface nodes; {len(interfaces.triangles):,} interface triangles')
print(f'Accepted Laplacian iterations: {interfaces.accepted_iterations}')
print('Frozen junction points:', np.count_nonzero(interfaces.node_kind == 3))
import json
print(json.dumps(interfaces.junction_metrics(), indent=2))

# Stable input for rerunning the optional point-contact separation cell.
interfaces_smoothed = interfaces


271,591 surface nodes; 558,440 interface triangles
Accepted Laplacian iterations: 50
Frozen junction points: 2725
{
  "movable_nodes": 20733,
  "junction_edges": 24518,
  "mean_turn_before_deg": 27.16538851106931,
  "mean_turn_after_deg": 12.804125890853921,
  "turns_over_45_before": 6258,
  "turns_over_45_after": 139,
  "maximum_line_displacement": 0.5229022228162987
}


### Check and stabilize smoothed geometry before Gmsh

Smoothing can introduce intersections even when voxel topology is valid. The guard checks triangle intersections, source-relative collapse/reversal and approach to RVE planes. Affected nodes and one neighbour ring are pulled back toward extracted voxel coordinates. Frozen junctions and existing RVE coordinates stay fixed. Local staircase detail may return where required; no triangles or grains are deleted.

Larger clearance keeps originally interior nodes farther from box planes (capped by their original distance). More passes permit more repair attempts. `GEOMETRY_ROLLBACK_RELAXATION` is the retained fraction of smoothing displacement per retry: **smaller values restore voxel detail more strongly**, larger values more gradually. A finite maximum correction bounds movement in physical units; None permits complete rollback. Disabled mode reports unverified geometry. A failed enabled repair raises.

The Gmsh cell additionally checks intersections/cap tangencies. Its bounded retries restore the source triangulation on affected charts, limiting geometric approximation there. These checks do not replace later closed-volume and tetrahedral-quality validation.


**Near-contact opening:** `MIN_FACET_OPENING_DEGREES` rejects almost closed wedges between triangles sharing an edge, including triple lines. Increasing it restores more local voxel detail and limits near-singular sheets; decreasing it permits sharper wedges. `0` disables this check. The default is 1 degree; Gmsh can reject angles below its 0.1 degree overlap tolerance even when there is no mathematical intersection. This control is checked before and after Gmsh and during quality repair. Frozen junction points and RVE-plane coordinates remain fixed.


In [7]:
# Restore local voxel detail where smoothing creates invalid geometry.
GUARD_SMOOTHED_GEOMETRY = True
MIN_FACET_OPENING_DEGREES = 1.0  # 0 disables; Gmsh rejects nearly closed wedges
GEOMETRY_BOUNDARY_CLEARANCE = 0.05 * float(np.min(effective_spacing))
GEOMETRY_MAX_PASSES = 12
GEOMETRY_ROLLBACK_RELAXATION = 0.5
GEOMETRY_MAX_CORRECTION = None  # None permits return to original voxel geometry.

_guard = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.geometry_guard'))
interfaces, geometry_report = _guard.stabilize_interfaces(
    interfaces_smoothed, np.array(labels_upscaled.shape) * effective_spacing,
    enabled=GUARD_SMOOTHED_GEOMETRY, boundary_clearance=GEOMETRY_BOUNDARY_CLEARANCE,
    max_passes=GEOMETRY_MAX_PASSES, relaxation=GEOMETRY_ROLLBACK_RELAXATION,
    max_correction=GEOMETRY_MAX_CORRECTION,
    minimum_facet_angle=MIN_FACET_OPENING_DEGREES)
GEOMETRY_AUDIT = RUN_DIR / 'CM03_geometry_guard.json'
GEOMETRY_AUDIT.write_text(json.dumps(geometry_report, indent=2))
print(json.dumps(geometry_report, indent=2))
interfaces_before_unpinch = interfaces


{
  "enabled": true,
  "verified": true,
  "remaining_intersections": 0,
  "moved_nodes": 6285,
  "maximum_correction": 1.3498621344625183,
  "boundary_clearance": 0.025,
  "history": [
    {
      "pass_number": 0,
      "intersections": 777,
      "small_facet_angles": 109,
      "affected_nodes": 1192
    },
    {
      "pass_number": 1,
      "intersections": 40,
      "small_facet_angles": 3,
      "affected_nodes": 945
    },
    {
      "pass_number": 2,
      "intersections": 16,
      "small_facet_angles": 1,
      "affected_nodes": 248
    },
    {
      "pass_number": 3,
      "intersections": 16,
      "small_facet_angles": 0,
      "affected_nodes": 135
    },
    {
      "pass_number": 4,
      "intersections": 3,
      "small_facet_angles": 0,
      "affected_nodes": 26
    },
    {
      "pass_number": 5,
      "intersections": 0,
      "small_facet_angles": 0,
      "affected_nodes": 10
    },
    {
      "pass_number": 6,
      "intersections": 0,
      "small_facet_a

## Separate point contacts and press down their tips before Gmsh
This optional step intentionally removes even legitimate point contacts when the incident triangles form disconnected surface fans. Each fan receives its own node and its tip is relaxed inward with a smoothly decaying influence over the surrounding surface. Topology can change, and grain volumes are not preserved.

`SEPARATE_PINCHES` turns this on/off. Radius, maximum movement, and minimum opening are in physical units; the defaults scale with the original voxel spacing, independently of upscaling. Only the nodes at selected contacts are released from their former common-node/frozen constraints. Other frozen junction points remain fixed, and RVE coordinates stay on their original planes.

The step preserves edge incidence, guards against triangle collapse and per-step reversal, and rolls back the batch if any duplicated tips cannot be opened sufficiently. Connected surface fans and shared junction edges are not cut; separating those would require reconstructing interface patches. It does not certify the absence of global intersections or produce a tetrahedral mesh.

Rerun the smoothing cell once to populate `interfaces_before_unpinch`. After that, this cell can be rerun with different settings without compounding its previous changes. The surface plot and Gmsh cells below use the resulting `interfaces`.


In [8]:
SEPARATE_PINCHES = True
PINCH_PRESS_RADIUS = 1.5 * min(SPACING)
PINCH_PRESS_MAX_DISPLACEMENT = 0.5 * min(SPACING)
PINCH_PRESS_ITERATIONS = 30
PINCH_PRESS_RELAXATION = 0.25
PINCH_MIN_OPENING = 0.02 * min(SPACING)

import json
_unpinch_module = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.unpinch'))
interfaces, unpinch_report, unpinch_audit = _unpinch_module.separate_pinch_fans(
    interfaces_before_unpinch, enabled=SEPARATE_PINCHES,
    radius=PINCH_PRESS_RADIUS, max_displacement=PINCH_PRESS_MAX_DISPLACEMENT,
    iterations=PINCH_PRESS_ITERATIONS, relaxation=PINCH_PRESS_RELAXATION,
    min_separation=PINCH_MIN_OPENING)
print(json.dumps(unpinch_report, indent=2))
UNPINCH_AUDIT = RUN_DIR / 'CM03_unpinch.json'
UNPINCH_AUDIT.parent.mkdir(parents=True, exist_ok=True)
UNPINCH_AUDIT.write_text(json.dumps({'report': unpinch_report, 'contacts': unpinch_audit}, indent=2))
print('Point-contact repair audit:', UNPINCH_AUDIT)


{
  "enabled": true,
  "candidate_contacts": 0,
  "split_contacts": 0,
  "stop_reason": "no disconnected surface fans"
}
Point-contact repair audit: c:\Development\UPXO\upxo_library\src\upxo\meshing\confMesh3D_01\output\CM03_unpinch.json


## Plot only the smoothed internal grain boundary surfaces
Opens an interactive PyVista window. Rotate and zoom to inspect the interfaces.
Set `GRAIN_ID` to a shuffled grain ID to isolate its internal interfaces, or leave it as `None` to show all interfaces. Each shared interface is coloured by its grain pair. No outer caps, tetrahedra or mesh edges are displayed by default.

In [9]:
import pyvista as pv

GRAIN_ID = None  # Example: int(np.unique(labels)[0]); IDs refer to shuffled labels.
SHOW_EDGES = False
SHOW_JUNCTION_LINES = False  # Overlay the actual junction graph.
OPACITY = 1.0

selected = (np.ones(len(interfaces.triangles), dtype=bool) if GRAIN_ID is None
            else np.any(interfaces.grain_pairs == GRAIN_ID, axis=1))
if not np.any(selected):
    print('No internal interfaces for this selection.')
else:
    triangles = interfaces.triangles[selected]
    faces = np.column_stack((np.full(len(triangles), 3), triangles)).ravel()
    boundary_surfaces = pv.PolyData(interfaces.points, faces)
    pairs = np.sort(interfaces.grain_pairs, axis=1)
    unique_pairs, pair_ids = np.unique(pairs, axis=0, return_inverse=True)
    pair_colours = np.random.default_rng(GRAIN_ID_SEED).permutation(len(unique_pairs))
    boundary_surfaces.cell_data['interface_colour'] = pair_colours[pair_ids[selected]]
    boundary_surfaces.cell_data['grain_a'] = pairs[selected, 0]
    boundary_surfaces.cell_data['grain_b'] = pairs[selected, 1]

    surface_plotter = pv.Plotter(notebook=False)
    surface_plotter.add_mesh(boundary_surfaces, scalars='interface_colour',
                             cmap='nipy_spectral', show_scalar_bar=False,
                             show_edges=SHOW_EDGES, opacity=OPACITY)
    if SHOW_JUNCTION_LINES and len(interfaces.junction_edges):
        line_edges = interfaces.junction_edges
        if GRAIN_ID is not None:
            visible_nodes = np.unique(triangles)
            line_edges = line_edges[np.all(np.isin(line_edges, visible_nodes), axis=1)]
        if len(line_edges):
            junction_lines = pv.PolyData(interfaces.points)
            junction_lines.lines = np.column_stack((np.full(len(line_edges), 2), line_edges)).ravel()
            surface_plotter.add_mesh(junction_lines, color='black', line_width=3)
    surface_plotter.add_text('Laplacian-smoothed grain interfaces', font_size=12)
    surface_plotter.add_axes()
    surface_plotter.show()


### Trace a small rounded feature to its voxel grain
No circular healing patches or point markers are inserted by the interface plot. Small retained grain regions can become rounded during smoothing, and contact separation can change their visibility. This audit traces a chosen original grain ID through colour shuffling and cleaning to its actual surface triangles. It does not remove small grains or assert that a rounded rendering is a physically spherical sub-block.


In [10]:
AUDIT_ORIGINAL_GRAIN_ID = None  # None selects a grain with the fewest input voxels.
SHOW_AUDIT_GRAIN = False

audit_ids, audit_counts = np.unique(labels_original, return_counts=True)
print('Input grain IDs occupying one voxel:', int(np.sum(audit_counts == 1)))
print('Input grain IDs occupying at most eight voxels:', int(np.sum(audit_counts <= 8)))
audit_original_id = (int(audit_ids[np.argmin(audit_counts)])
                     if AUDIT_ORIGINAL_GRAIN_ID is None else int(AUDIT_ORIGINAL_GRAIN_ID))
audit_display_id = grain_id_map[audit_original_id]
audit_triangles = interfaces.triangles[np.any(interfaces.grain_pairs == audit_display_id, axis=1)]
print('Original ID:', audit_original_id, '| shuffled/display ID:', audit_display_id)
print('Original voxels:', int(np.count_nonzero(labels_original == audit_original_id)))
print('Voxels after cleaning (before upscaling):',
      int(np.count_nonzero(labels_meshing == audit_display_id)))
print('Internal interface triangles:', len(audit_triangles))
if SHOW_AUDIT_GRAIN and len(audit_triangles):
    audit_nodes, audit_conn = np.unique(audit_triangles, return_inverse=True)
    audit_surface = pv.PolyData(interfaces.points[audit_nodes],
        np.column_stack((np.full(len(audit_triangles), 3), audit_conn.reshape(-1, 3))).ravel())
    audit_plotter = pv.Plotter(notebook=False)
    audit_plotter.add_mesh(audit_surface, color='orange', show_edges=True)
    audit_plotter.add_text(f'Original grain {audit_original_id} | display ID {audit_display_id}', font_size=11)
    audit_plotter.add_axes()
    audit_plotter.show()


Input grain IDs occupying one voxel: 56
Input grain IDs occupying at most eight voxels: 185
Original ID: 14 | shuffled/display ID: 165
Original voxels: 1
Voxels after cleaning (before upscaling): 1
Internal interface triangles: 48


## Gmsh remeshing with connected interface boundaries
Gmsh builds shared discrete curves from the labelled surface topology, parametrizes each surface, and regenerates its 2D triangles. Existing curve nodes and edges are retained: neighboring interfaces therefore share the same boundary discretization, and traces on RVE faces stay on those faces at their smoothed positions. Frozen junction points also remain fixed. No volume mesh or exterior caps are generated.

`GMSH_SIZE` is the target interior triangle size in physical units; fixed boundary segments retain their existing lengths. Parametrization can split a grain-pair interface into several charts. The step checks every chart boundary and preserves all grain-pair labels; failures raise an error instead of silently skipping an interface. The original `interfaces` object is unchanged.

Uses Gmsh's [discrete geometry and topology APIs](https://gmsh.info/doc/texinfo/gmsh.html#Namespace-gmsh_002fmodel_002fmesh), with `Mesh.MeshOnlyEmpty` to retain the curve mesh. Requires `gmsh` and `pyvista` in the selected kernel (verified with Gmsh 4.14.1). The VTP export retains both grain IDs for every triangle.


Change `GMSH_ALGORITHM` to compare triangular surface meshers: 1 MeshAdapt, 2 Automatic, 5 Delaunay, 6 Frontal-Delaunay (the previous default). Keep geometry and `GMSH_SIZE` fixed when comparing. Gmsh can fall back internally to MeshAdapt; use `GMSH_VERBOSE = True` to inspect its messages. The report records the requested algorithm, not a guarantee of which fallback ran. Shared-curve and RVE constraints remain checked for every option. Changing the algorithm does not repair a distorted underlying parametrization.


**Gmsh patch size:** `GMSH_CHART_TRIANGLES` bounds the number of source triangles in each parametrization patch. Decrease it for smaller, more constrained patches; increase it for more freedom to remesh across the surface. Extra patch boundaries remain shared, so they do not create gaps. `None` uses automatic topology partitioning. Intersection checks remain necessary regardless of this setting.


In [11]:
# Regenerate all grain-interface surface triangles in Gmsh.
import importlib
import json
import pyvista as pv

GMSH_SIZE = 0.75 * min(SPACING)  # Physical target size; boundary curves stay fixed.
GMSH_ALGORITHM = 6  # 1: MeshAdapt; 2: Automatic; 5: Delaunay; 6: Frontal-Delaunay
GMSH_CHECK_INTERSECTIONS = True
GMSH_INTERSECTION_RETRIES = 4
GMSH_CHART_TRIANGLES = 128  # Maximum source triangles per parametrization patch
GMSH_VERBOSE = False
GMSH_SHOW_CURVES = False

_gmsh_remesher = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.gmsh_interfaces'))
gmsh_result = _gmsh_remesher.remesh_interfaces_gmsh(
    interfaces, mesh_size=GMSH_SIZE, verbose=GMSH_VERBOSE,
    algorithm=GMSH_ALGORITHM, check_intersections=GMSH_CHECK_INTERSECTIONS,
    intersection_retries=GMSH_INTERSECTION_RETRIES,
    max_chart_triangles=GMSH_CHART_TRIANGLES,
    minimum_facet_angle=MIN_FACET_OPENING_DEGREES,
    rve_dimensions=np.array(labels_upscaled.shape) * effective_spacing)
print(json.dumps(gmsh_result.report, indent=2))

cells = np.column_stack((np.full(len(gmsh_result.triangles), 3), gmsh_result.triangles)).ravel()
gmsh_surfaces = pv.PolyData(gmsh_result.points, cells)
gmsh_surfaces.cell_data['grain_a'] = gmsh_result.grain_pairs[:, 0]
gmsh_surfaces.cell_data['grain_b'] = gmsh_result.grain_pairs[:, 1]
remeshed_pairs, remeshed_pair_ids = np.unique(gmsh_result.grain_pairs, axis=0, return_inverse=True)
pair_colours = np.random.default_rng(GRAIN_ID_SEED).permutation(len(remeshed_pairs))
gmsh_surfaces.cell_data['interface_colour'] = pair_colours[remeshed_pair_ids]

GMSH_OUTPUT = RUN_DIR / 'CM03_gmsh.vtp'
GMSH_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
gmsh_surfaces.save(GMSH_OUTPUT)
GMSH_OUTPUT.with_suffix('.json').write_text(json.dumps(gmsh_result.report, indent=2))
print('Saved:', GMSH_OUTPUT)

{
  "input_triangles": 558440,
  "output_triangles": 763452,
  "requested_algorithm": 6,
  "intersections_checked": true,
  "remaining_intersections": 0,
  "remaining_small_facet_angles": 0,
  "minimum_facet_angle": 1.0,
  "source_chart_orientation_verified": true,
  "reoriented_surface_charts": [
    361,
    2193,
    2350,
    11526
  ],
  "grain_pairs": 1617,
  "surface_charts": 17152,
  "preserved_curve_nodes": 135721,
  "shared_boundaries_verified": true,
  "curve_coordinates_preserved": true,
  "mesh_size": 0.75,
  "rve_trace_nodes_preserved": 10055,
  "frozen_junction_points_preserved": 2725,
  "surface_meshing_warnings": [],
  "surface_warning_capture_available": true,
  "maximum_curve_coordinate_error": 0.0,
  "geometric_repairs": [
    {
      "detected_intersections": 104,
      "small_facet_angles": 290,
      "cap_tangencies": 8,
      "restored_source_charts": 167
    }
  ],
  "parametrization_repairs": [
    {
      "stage": "surface_generation",
      "replacement_disk

In [12]:
gmsh_plotter = pv.Plotter(notebook=False)
gmsh_plotter.add_mesh(gmsh_surfaces, scalars='interface_colour', cmap='nipy_spectral',
                     show_edges=False, edge_color='gray', show_scalar_bar=False)
if GMSH_SHOW_CURVES and len(gmsh_result.curve_edges):
    preserved_curves = pv.PolyData(gmsh_result.points)
    preserved_curves.lines = np.column_stack((np.full(len(gmsh_result.curve_edges), 2),
                                              gmsh_result.curve_edges)).ravel()
    gmsh_plotter.add_mesh(preserved_curves, color='black', line_width=2)
gmsh_plotter.add_text('Gmsh-remeshed interfaces | shared curves fixed', font_size=11)
gmsh_plotter.add_axes()
gmsh_plotter.show()


## Close all six RVE faces
Build grain-labelled planar caps from the current Gmsh interface traces and the RVE perimeter. Boundary segments reuse the existing interface node IDs without moving them or subdividing their edges. Caps are triangulated by Gmsh and joined into one surface complex.

This cell checks complete face coverage, a closed exterior shell, per-grain edge closure, positive enclosed signed grain volumes and total RVE volume. Existing nonmanifold grain edges are reported separately; capping does not repair those or certify absence of self-intersections. No tetrahedra are generated here.

Run the preceding Gmsh cell first. `CAP_MESH_SIZE` controls cap-interior resolution; boundary segments remain fixed. `SHOW_INTERNAL_INTERFACES` can overlay the internal surfaces, and reducing `CAP_OPACITY` reveals them. The complete closed surface complex, the exterior caps alone, and a validation report are exported. On cap triangles `grain_a == grain_b`; `is_rve_cap` distinguishes them from internal interfaces. Face IDs 0..5 mean x-min, x-max, y-min, y-max, z-min, z-max.

Box perimeter edges are subdivided using `RVE_EDGE_SIZE`; adjacent RVE faces share these nodes. Rerun this closure cell, then the final joint remeshing cell after changing the size.


In [13]:
import importlib
import json
import pyvista as pv

CAP_MESH_SIZE = GMSH_SIZE
RVE_EDGE_SIZE = CAP_MESH_SIZE  # Maximum new box-edge segment length
CAP_OPACITY = 1.0
SHOW_INTERNAL_INTERFACES = False
CAP_SHOW_EDGES = True

_rve_caps = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.rve_caps'))
closed_rve = _rve_caps.close_rve_faces(
    gmsh_result, labels_upscaled, spacing=effective_spacing,
    mesh_size=CAP_MESH_SIZE, edge_size=RVE_EDGE_SIZE)
print(json.dumps({k: v for k, v in closed_rve.report.items()
                  if k != 'enclosed_grain_volumes'}, indent=2))

closed_cells = np.column_stack((np.full(len(closed_rve.triangles), 3),
                                closed_rve.triangles)).ravel()
closed_rve_surface = pv.PolyData(closed_rve.points, closed_cells)
closed_rve_surface.cell_data['grain_a'] = closed_rve.grain_pairs[:, 0]
closed_rve_surface.cell_data['grain_b'] = closed_rve.grain_pairs[:, 1]
closed_rve_surface.cell_data['is_rve_cap'] = closed_rve.exterior.astype(np.uint8)
closed_rve_surface.cell_data['rve_face'] = closed_rve.rve_face
rve_caps_surface = closed_rve_surface.extract_cells(closed_rve.exterior).extract_surface()

CLOSED_RVE_OUTPUT = RUN_DIR / 'CM03_closed_RVE.vtp'
CLOSED_RVE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
closed_rve_surface.save(CLOSED_RVE_OUTPUT)
rve_caps_surface.save(CLOSED_RVE_OUTPUT.with_name('CM03_RVE_caps.vtp'))
CLOSED_RVE_OUTPUT.with_suffix('.json').write_text(json.dumps(closed_rve.report, indent=2))
print('Saved closed surface complex:', CLOSED_RVE_OUTPUT)

rve_plotter = pv.Plotter(notebook=False)
rve_plotter.add_mesh(rve_caps_surface, scalars='grain_a', cmap='nipy_spectral',
                     show_edges=CAP_SHOW_EDGES, edge_color='gray', opacity=CAP_OPACITY,
                     show_scalar_bar=False)
if SHOW_INTERNAL_INTERFACES:
    rve_plotter.add_mesh(gmsh_surfaces, scalars='grain_a', cmap='nipy_spectral',
                         show_edges=False, show_scalar_bar=False)
rve_plotter.add_text('Closed RVE faces | grain-labelled caps', font_size=12)
rve_plotter.add_axes()
rve_plotter.show()


{
  "grains": 373,
  "cap_regions": 299,
  "rve_edge_size": 0.75,
  "subdivided_rve_edge_segments": 108,
  "added_rve_edge_nodes": 741,
  "cap_triangles": 239636,
  "total_triangles": 1003088,
  "all_six_faces_closed": true,
  "grain_surface_edge_closure_verified": true,
  "grains_with_nonmanifold_edges": [],
  "rve_dimensions": [
    50.0,
    50.0,
    50.0
  ],
  "rve_volume": 125000.0,
  "source_interface_nodes_unchanged": true
}
Saved closed surface complex: c:\Development\UPXO\upxo_library\src\upxo\meshing\confMesh3D_01\output\CM03_closed_RVE.vtp


### Joint Gmsh remeshing: RVE caps and internal interfaces
Run after the RVE closure cell. All labelled surface patches are regenerated together. Shared curve nodes/edges stay fixed; cap interior nodes are remeshed on their RVE planes. This is surface meshing only. The report checks closure and lists any nonmanifold grain edges that still require repair before tetrahedral meshing.


### Remove surface slivers before the joint remesh

`PRE_JOINT_QUALITY_REPAIR` enables guarded diagonal flips before Gmsh sees the closed surface. This reduces nearly collinear input triangles that can cause parametrization failures. `PRE_JOINT_QUALITY_TARGET` (0–1) targets more triangles when increased; `PRE_JOINT_QUALITY_MAX_PASSES` allows more attempts at additional cost. Node coordinates and grain/RVE patch boundaries remain fixed; local curved shape and grain volumes can change. The later quality-repair cell performs the same checks on the final remeshed result.


In [14]:
PRE_JOINT_QUALITY_REPAIR = True
PRE_JOINT_QUALITY_TARGET = 0.05
PRE_JOINT_QUALITY_MAX_PASSES = 10

_quality = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.surface_quality'))
closed_rve = _quality.improve_surface_quality(
    closed_rve, enabled=PRE_JOINT_QUALITY_REPAIR,
    minimum_quality=PRE_JOINT_QUALITY_TARGET, max_passes=PRE_JOINT_QUALITY_MAX_PASSES,
    minimum_facet_angle=MIN_FACET_OPENING_DEGREES)
print(json.dumps(closed_rve.report['surface_quality_repair'], indent=2))
CLOSED_RVE_OUTPUT.with_name('CM03_pre_joint_quality.json').write_text(
    json.dumps(closed_rve.report['surface_quality_repair'], indent=2))


{
  "enabled": true,
  "threshold": 0.05,
  "minimum_facet_angle": 1.0,
  "before": {
    "minimum": 1.0511491412408772e-06,
    "poor_count": 191
  },
  "after": {
    "minimum": 0.01814217190660665,
    "poor_count": 2
  },
  "history": [
    {
      "pass_number": 1,
      "proposed": 166,
      "accepted": 166,
      "rejected_geometry": 0
    },
    {
      "pass_number": 2,
      "proposed": 21,
      "accepted": 21,
      "rejected_geometry": 0
    },
    {
      "pass_number": 3,
      "proposed": 9,
      "accepted": 9,
      "rejected_geometry": 0
    },
    {
      "pass_number": 4,
      "proposed": 3,
      "accepted": 3,
      "rejected_geometry": 0
    },
    {
      "pass_number": 5,
      "proposed": 2,
      "accepted": 2,
      "rejected_geometry": 0
    },
    {
      "pass_number": 6,
      "proposed": 2,
      "accepted": 2,
      "rejected_geometry": 0
    }
  ],
  "accepted_flips": 203,
  "node_coordinates_unchanged": true,
  "full_validation_required": true
}


998

In [15]:
# Remesh all caps and internal interfaces in one Gmsh model.
FINAL_GMSH_SIZE = GMSH_SIZE
FINAL_GMSH_ALGORITHM = GMSH_ALGORITHM  # 1, 2, 5, or 6
FINAL_SHOW_INTERNAL = False

from importlib import import_module, reload
reload(import_module('upxo.meshing.gbconformant.d3v2p0.gmsh_interfaces'))
_joint = reload(import_module('upxo.meshing.gbconformant.d3v2p0.gmsh_closed'))
remeshed_rve = _joint.remesh_closed_rve_gmsh(
    closed_rve, mesh_size=FINAL_GMSH_SIZE, algorithm=FINAL_GMSH_ALGORITHM,
    check_intersections=GMSH_CHECK_INTERSECTIONS, intersection_retries=GMSH_INTERSECTION_RETRIES,
    max_chart_triangles=GMSH_CHART_TRIANGLES,
    minimum_facet_angle=MIN_FACET_OPENING_DEGREES)
print({k: v for k, v in remeshed_rve.report.items()
       if k != 'enclosed_grain_volumes'})

remeshed_rve_surface = pv.PolyData(
    remeshed_rve.points,
    np.column_stack((np.full(len(remeshed_rve.triangles), 3),
                     remeshed_rve.triangles)).ravel())
for key, values in dict(grain_a=remeshed_rve.grain_pairs[:, 0],
                        grain_b=remeshed_rve.grain_pairs[:, 1],
                        is_rve_cap=remeshed_rve.exterior.astype(np.uint8),
                        rve_face=remeshed_rve.rve_face).items():
    remeshed_rve_surface.cell_data[key] = values
remeshed_rve_caps = remeshed_rve_surface.extract_cells(
    remeshed_rve.exterior).extract_surface()
FINAL_RVE_OUTPUT = CLOSED_RVE_OUTPUT.with_name('CM03_closed_RVE_remeshed.vtp')
remeshed_rve_surface.save(FINAL_RVE_OUTPUT)
import json
FINAL_RVE_OUTPUT.with_suffix('.json').write_text(
    json.dumps(remeshed_rve.report, indent=2))
final_plotter = pv.Plotter(notebook=False)
final_plotter.add_mesh(remeshed_rve_caps, scalars='grain_a', cmap='nipy_spectral',
                       show_edges=True, opacity=0.35 if FINAL_SHOW_INTERNAL else 1.)
if FINAL_SHOW_INTERNAL:
    final_plotter.add_mesh(remeshed_rve_surface.extract_cells(~remeshed_rve.exterior),
                           scalars='grain_a', cmap='nipy_spectral', show_edges=True)
final_plotter.add_axes()
final_plotter.show()


{'input_triangles': 1003088, 'output_triangles': 1070346, 'requested_algorithm': 6, 'intersections_checked': True, 'remaining_intersections': 0, 'remaining_small_facet_angles': 0, 'minimum_facet_angle': 1.0, 'source_chart_orientation_verified': True, 'reoriented_surface_charts': [1186, 2744, 28050, 29932, 34842], 'surface_charts': 39184, 'preserved_curve_nodes': 284876, 'shared_boundaries_verified': True, 'curve_coordinates_preserved': True, 'mesh_size': 0.75, 'rve_trace_nodes_preserved': 10804, 'frozen_junction_points_preserved': 33895, 'surface_meshing_warnings': [], 'surface_warning_capture_available': True, 'maximum_curve_coordinate_error': 0.0, 'geometric_repairs': [{'detected_intersections': 237, 'small_facet_angles': 499, 'cap_tangencies': 13, 'restored_source_charts': 260}, {'detected_intersections': 0, 'small_facet_angles': 1, 'cap_tangencies': 0, 'restored_source_charts': 2}], 'parametrization_repairs': [{'stage': 'surface_generation', 'replacement_disk_charts': 1448, 'source

### Pre-tetrahedral-meshing surface validation
Run after the final joint surface remeshing cell. This read-only check reports per-grain closure, edge/vertex manifoldness, orientation, ownership, duplicate triangles/nodes, triangle quality, RVE planes/areas and signed volumes. Shared triple lines are checked per grain. `BLOCKED` means defects were found. A preliminary pass is **not** a guarantee of successful tetrahedral meshing: triangle intersections, clearance and shell containment remain unchecked. No tetrahedra are generated.


In [16]:
# Validate the final surface complex before attempting Gmsh volume meshing.
MIN_SURFACE_TRIANGLE_QUALITY = 0.05
PLOT_VALIDATION_ISSUES = True
STOP_ON_VALIDATION_ERRORS = False  # True raises after saving the report.

from importlib import import_module, reload
import json
_validation = reload(import_module('upxo.meshing.gbconformant.d3v2p0.tet_validation'))
tet_validation = _validation.validate_tet_surfaces(
    remeshed_rve, expected_grains=np.unique(labels_upscaled),
    min_triangle_quality=MIN_SURFACE_TRIANGLE_QUALITY, check_intersections=True, minimum_facet_angle=MIN_FACET_OPENING_DEGREES)
print('Pre-tet validation:', tet_validation['status'])
print('Triangle quality:', tet_validation.get('triangle_quality'))
for category in ('blockers', 'warnings', 'unchecked'):
    print('\n' + category.upper())
    for message in tet_validation[category]:
        print(' -', message)
print('\nPer-grain defects:')
for grain, checks in tet_validation['per_grain'].items():
    if any(checks[k] for k in ('open_edges', 'nonmanifold_edges',
                               'inconsistent_oriented_edges', 'nonmanifold_vertices')):
        print(grain, checks)
VALIDATION_OUTPUT = FINAL_RVE_OUTPUT.with_name('CM03_pre_tet_validation.json')
VALIDATION_OUTPUT.write_text(json.dumps(tet_validation, indent=2))
print('Saved:', VALIDATION_OUTPUT)

if PLOT_VALIDATION_ISSUES and tet_validation['bad_triangle_ids']:
    issue_plotter = pv.Plotter(notebook=False)
    issue_plotter.add_mesh(remeshed_rve_surface, color='lightgray', opacity=0.1)
    issue_plotter.add_mesh(remeshed_rve_surface.extract_cells(
        np.asarray(tet_validation['bad_triangle_ids'], dtype=int)),
        color='red', show_edges=True)
    issue_plotter.add_text('Red: topology/orientation defects or poor triangles', font_size=10)
    issue_plotter.add_axes()
    issue_plotter.show()
if STOP_ON_VALIDATION_ERRORS and tet_validation['blockers']:
    raise RuntimeError('Surface validation blocked: inspect tet_validation and its JSON report.')


Pre-tet validation: PRELIMINARY_CHECKS_PASSED
Triangle quality: {'minimum': 0.00015922692144869227, 'percentiles_1_5_50': [0.43892337821479094, 0.6318539011100999, 0.9450925395822104], 'poor_count': 298, 'threshold': 0.05, 'definition': '4*sqrt(3)*area / sum(squared edge lengths); equilateral=1'}

BLOCKERS

WARNINGS
 - 298 triangles below quality threshold 0.05
 - Grain 153: 2 separate shells; containment needs checking
 - Grain 184: 2 separate shells; containment needs checking
 - Grain 185: 2 separate shells; containment needs checking
 - Grain 204: 2 separate shells; containment needs checking
 - Grain 279: 3 separate shells; containment needs checking
 - Grain 293: 2 separate shells; containment needs checking
 - Grain 301: 4 separate shells; containment needs checking
 - Grain 330: 2 separate shells; containment needs checking
 - Grain 343: 2 separate shells; containment needs checking

UNCHECKED
 - minimum separation between nonadjacent surfaces
 - nested-shell containment and te

### Repair contacts, revalidate, and verify tetrahedral meshing
These cells use the final `remeshed_rve` without altering it. The repair separates disconnected sheets at point/edge contacts and moves only those contact nodes by a bounded amount. True shared junctions remain connected and box-plane coordinates remain fixed. This deliberately changes local grain morphology. Unsupported defects stop the subsequent volume-meshing cell.

The final cell generates a real Gmsh tetrahedral mesh, including disconnected pieces with their original grain IDs. It checks positive element volumes, every grain's surface conformity and volume, and minimum tetrahedron quality. `TET_MESH_VERIFIED` applies to the generated mesh and selected quality threshold; it is not a universal guarantee for different settings or future datasets.

The intersection-repair cell separates crossing interface wedges while freezing their shared junctions. Final validation and the tetrahedral mesher both test geometric triangle intersections, including crossings between triangles that share a node. These are tolerance-based checks; minimum-clearance certification remains separate.


In [17]:
from importlib import import_module, reload
import json

REPAIR_CONTACTS = True
REPAIR_SEPARATION = 0.15 * float(np.min(SPACING))  # Maximum movement in physical units
SHOW_REPAIRED_SURFACES = True

_repair = reload(import_module('upxo.meshing.gbconformant.d3v2p0.tet_repair'))
repaired_rve = (_repair.repair_tet_contacts(remeshed_rve, separation=REPAIR_SEPARATION)
                if REPAIR_CONTACTS else remeshed_rve)
print('Contact repair:', repaired_rve.report.get('contact_repair', 'disabled'))
repaired_rve_surface = pv.PolyData(
    repaired_rve.points,
    np.column_stack((np.full(len(repaired_rve.triangles), 3), repaired_rve.triangles)).ravel())
repaired_rve_surface.cell_data['grain_a'] = repaired_rve.grain_pairs[:, 0]
repaired_rve_surface.cell_data['grain_b'] = repaired_rve.grain_pairs[:, 1]
repaired_rve_surface.cell_data['is_rve_cap'] = repaired_rve.exterior.astype(np.uint8)
repaired_rve_surface.cell_data['rve_face'] = repaired_rve.rve_face
REPAIRED_OUTPUT = FINAL_RVE_OUTPUT.with_name('CM03_repaired_RVE.vtp')
repaired_rve_surface.save(REPAIRED_OUTPUT)
if SHOW_REPAIRED_SURFACES:
    repair_plotter = pv.Plotter(notebook=False)
    repair_plotter.add_mesh(repaired_rve_surface, scalars='grain_a', cmap='nipy_spectral',
                             show_edges=True, opacity=0.5)
    moved = repaired_rve.report.get('contact_repair', {}).get('split_source_nodes', [])
    if moved:
        repair_plotter.add_points(remeshed_rve.points[moved], color='red',
                                  point_size=12, render_points_as_spheres=True)
    repair_plotter.add_axes()
    repair_plotter.show()


Contact repair: {'split_source_nodes': [], 'added_nodes': 0, 'separation_limit': 0.15, 'separable_contact_edges': 0}


In [18]:
# Remove geometric crossings beside junction lines before the final checks.
REPAIR_INTERFACE_CROSSINGS = True
CROSSING_MAX_DISPLACEMENT = 0.35 * float(np.min(SPACING))
CROSSING_NEIGHBOURHOOD_RADIUS = 1.0 * float(np.min(SPACING))
CROSSING_OPENING = 0.10  # Separation relative to distance from the junction line

_crossings = reload(import_module('upxo.meshing.gbconformant.d3v2p0.intersection_repair'))
if REPAIR_INTERFACE_CROSSINGS:
    repaired_rve = _crossings.repair_interface_crossings(
        repaired_rve, max_displacement=CROSSING_MAX_DISPLACEMENT,
        radius=CROSSING_NEIGHBOURHOOD_RADIUS, opening=CROSSING_OPENING)
    print('Intersection repair:', repaired_rve.report['intersection_repair'])
    repaired_rve_surface.points = repaired_rve.points
    repaired_rve_surface.save(REPAIRED_OUTPUT)
else:
    print('Intersection repair disabled; the following checks still detect crossings.')


Intersection repair: {'initial_intersections': 0, 'remaining_intersections': 0, 'moved_nodes': [], 'maximum_displacement': 0.0, 'junction_nodes_fixed': True, 'passes': 0}


### Improve poor surface triangles

`REPAIR_SURFACE_QUALITY` turns guarded diagonal flips on/off. Node coordinates, grain junction edges and RVE patch boundaries stay fixed. A flip must improve the minimum quality of its two triangles and pass intersection checks against the full complex. Flips on curved patches may change local surface shape and individual grain volumes; the shared complex remains connected.

`SURFACE_QUALITY_TARGET` uses the same 0–1 triangle metric as validation. Increase it to target more triangles; decrease it to accept more elongated triangles. `SURFACE_QUALITY_MAX_PASSES` allows more local attempts when increased, at additional cost. Some constrained triangles cannot be improved by flips; their remaining count is reported. This does not guarantee the tetrahedral quality target. The following full validation remains required.


In [19]:
REPAIR_SURFACE_QUALITY = True
SURFACE_QUALITY_TARGET = MIN_SURFACE_TRIANGLE_QUALITY
SURFACE_QUALITY_MAX_PASSES = 10

_quality = reload(import_module('upxo.meshing.gbconformant.d3v2p0.surface_quality'))
repaired_rve = _quality.improve_surface_quality(
    repaired_rve, enabled=REPAIR_SURFACE_QUALITY,
    minimum_quality=SURFACE_QUALITY_TARGET, max_passes=SURFACE_QUALITY_MAX_PASSES,
    minimum_facet_angle=MIN_FACET_OPENING_DEGREES)
print(json.dumps(repaired_rve.report['surface_quality_repair'], indent=2))
repaired_rve_surface = pv.PolyData(
    repaired_rve.points,
    np.column_stack((np.full(len(repaired_rve.triangles), 3), repaired_rve.triangles)).ravel())
for key, values in dict(grain_a=repaired_rve.grain_pairs[:, 0],
                        grain_b=repaired_rve.grain_pairs[:, 1],
                        is_rve_cap=repaired_rve.exterior.astype(np.uint8),
                        rve_face=repaired_rve.rve_face).items():
    repaired_rve_surface.cell_data[key] = values
repaired_rve_surface.save(REPAIRED_OUTPUT)


{
  "enabled": true,
  "threshold": 0.05,
  "minimum_facet_angle": 1.0,
  "before": {
    "minimum": 0.00015922692144869227,
    "poor_count": 298
  },
  "after": {
    "minimum": 0.014880802603587493,
    "poor_count": 3
  },
  "history": [
    {
      "pass_number": 1,
      "proposed": 281,
      "accepted": 281,
      "rejected_geometry": 0
    },
    {
      "pass_number": 2,
      "proposed": 15,
      "accepted": 15,
      "rejected_geometry": 0
    },
    {
      "pass_number": 3,
      "proposed": 2,
      "accepted": 2,
      "rejected_geometry": 0
    }
  ],
  "accepted_flips": 298,
  "node_coordinates_unchanged": true,
  "full_validation_required": true
}


In [20]:
# A fresh report is required: the pre-repair report is not reused.
_validation = reload(import_module('upxo.meshing.gbconformant.d3v2p0.tet_validation'))
repaired_validation = _validation.validate_tet_surfaces(
    repaired_rve, expected_grains=np.unique(labels_upscaled),
    min_triangle_quality=MIN_SURFACE_TRIANGLE_QUALITY, check_intersections=True, minimum_facet_angle=MIN_FACET_OPENING_DEGREES)
print('After repair:', repaired_validation['status'])
for category in ('blockers', 'warnings', 'unchecked'):
    print(category.upper(), repaired_validation[category])
print('Surface quality:', repaired_validation.get('triangle_quality'))
REPAIRED_OUTPUT.with_suffix('.json').write_text(json.dumps({
    'repair': repaired_rve.report.get('contact_repair'),
    'surface_quality_repair': repaired_rve.report.get('surface_quality_repair'),
    'intersection_repair': repaired_rve.report.get('intersection_repair'),
    'validation': repaired_validation}, indent=2))
if repaired_validation['blockers']:
    raise RuntimeError('Repair has not resolved all surface defects; inspect repaired_validation.')


After repair: PRELIMINARY_CHECKS_PASSED
BLOCKERS []
WARNINGS ['3 triangles below quality threshold 0.05', 'Grain 153: 2 separate shells; containment needs checking', 'Grain 184: 2 separate shells; containment needs checking', 'Grain 185: 2 separate shells; containment needs checking', 'Grain 204: 2 separate shells; containment needs checking', 'Grain 279: 3 separate shells; containment needs checking', 'Grain 293: 2 separate shells; containment needs checking', 'Grain 301: 4 separate shells; containment needs checking', 'Grain 330: 2 separate shells; containment needs checking', 'Grain 343: 2 separate shells; containment needs checking']
UNCHECKED ['minimum separation between nonadjacent surfaces', 'nested-shell containment and tetrahedron quality']
Surface quality: {'minimum': 0.014880802603587493, 'percentiles_1_5_50': [0.4411549979357462, 0.6322208464899015, 0.945106432599977], 'poor_count': 3, 'threshold': 0.05, 'definition': '4*sqrt(3)*area / sum(squared edge lengths); equilateral

In [21]:
# Generate and verify tetrahedra; readiness is based on this actual result.
RUN_TET_VERIFICATION = True
TET_MESH_SIZE = 1.5 * float(np.min(SPACING))
TET_OPTIMIZE_NETGEN = True  # Keep baseline if optimization worsens a volume
MIN_TET_QUALITY = 0.05  # Gmsh minSICN: larger positive values are better
SHOW_TET_CRINKLE_CLIP = True

tet_mesh_ready = False
verified_tets = None
if RUN_TET_VERIFICATION:
    reload(import_module('upxo.meshing.gbconformant.d3v2p0.tet_validation'))
    _tets = reload(import_module('upxo.meshing.gbconformant.d3v2p0.gmsh_tets'))
    # Rechecked inside the mesher, even if notebook cells were run out of order.
    verified_tets = _tets.mesh_repaired_rve_gmsh(
        repaired_rve, mesh_size=TET_MESH_SIZE, optimize_netgen=TET_OPTIMIZE_NETGEN,
        minimum_quality=MIN_TET_QUALITY)
    tet_mesh_ready = verified_tets.report['ready']
    print(json.dumps(verified_tets.report, indent=2))
    tet_grid = pv.UnstructuredGrid(
        np.column_stack((np.full(len(verified_tets.tetrahedra), 4),
                         verified_tets.tetrahedra)).ravel(),
        np.full(len(verified_tets.tetrahedra), pv.CellType.TETRA, dtype=np.uint8),
        verified_tets.points)
    tet_grid.cell_data['grain_id'] = verified_tets.grain_ids
    tet_grid.cell_data['minSICN'] = verified_tets.quality
    TET_OUTPUT = FINAL_RVE_OUTPUT.with_name('CM03_verified_tets.vtu')
    tet_grid.save(TET_OUTPUT)
    TET_OUTPUT.with_suffix('.json').write_text(json.dumps(verified_tets.report, indent=2))
    if SHOW_TET_CRINKLE_CLIP:
        tet_plotter = pv.Plotter(notebook=False)
        tet_plotter.add_mesh_clip_plane(tet_grid, scalars='grain_id', cmap='tab20c',
                                        show_edges=True, crinkle=True, normal='x')
        tet_plotter.add_axes()
        tet_plotter.show()
    if not tet_mesh_ready:
        print('Tetrahedra generated, but the requested quality threshold was not met.')
else:
    print('Tet verification skipped; tet_mesh_ready remains False.')


{
  "status": "TET_QUALITY_BELOW_TARGET",
  "ready": false,
  "grains": 373,
  "volume_entities": 385,
  "tetrahedra": 6943047,
  "nodes": 1249295,
  "per_grain": {
    "1": {
      "tetrahedra": 60014,
      "volume": 1041.4180754912393,
      "minimum_quality": 0.06967830668802015
    },
    "2": {
      "tetrahedra": 58,
      "volume": 0.0015187498187637864,
      "minimum_quality": 0.02144629396420481
    },
    "3": {
      "tetrahedra": 41684,
      "volume": 776.689699226672,
      "minimum_quality": 0.09153976686653811
    },
    "4": {
      "tetrahedra": 101820,
      "volume": 2071.091612982635,
      "minimum_quality": 0.07244720498110001
    },
    "5": {
      "tetrahedra": 58,
      "volume": 0.0015187519666799859,
      "minimum_quality": 0.021446289781093763
    },
    "6": {
      "tetrahedra": 122173,
      "volume": 2189.3237953762846,
      "minimum_quality": 0.050154196498040725
    },
    "7": {
      "tetrahedra": 17188,
      "volume": 285.09907195330493,
    

In [22]:
tet_plotter = pv.Plotter(notebook=False)
tet_plotter.add_mesh_clip_plane(tet_grid, scalars='grain_id', cmap='tab20c',
                                show_edges=False, crinkle=True, normal='x')
tet_plotter.add_axes()
tet_plotter.show()

## Verified 2× sample run and remaining quality limitations

The tested 50³ `sblk_lgi.npy` sample retained all **373 grain IDs**. Targeted voxel cleaning reduced 668 cubical boundary-link defects to zero. The guarded and jointly remeshed closed surfaces passed the intersection and 1° minimum facet-opening checks, with all six RVE faces closed.

Actual volume meshing produced **6,942,599 tetrahedra** and **1,249,221 nodes**. Every tetrahedron had positive volume, every grain boundary matched the repaired source triangles, and the total volume was 125,000. These are results for this saved run, not guarantees for changed inputs or controls.

The requested minSICN target **0.05 remains unmet**: 5,139 tetrahedra (0.0740%) fall below it; the minimum is 0.009606 and the median is 0.85525. The notebook correctly keeps `tet_mesh_ready = False`. Of those poor elements, 5,122 have all four nodes on the fixed surface. Many occur in very small retained grains substantially contracted by smoothing. Interior-node optimization alone cannot move those boundary nodes. A further quality improvement should preserve small-grain size and improve their surface geometry; 3× refinement alone is not a guaranteed remedy.

Use the optional view below to inspect the current run's low-quality elements. Raising `QUALITY_VIEW_THRESHOLD` shows more elements; lowering it isolates the worst. This display threshold does not change meshing or acceptance criteria. The saved audits are in `confMesh3D_01/output/CM03_verified_tets.json` and `CM03_quality_tail.json`. Restart the kernel and run from the top when changing preprocessing or smoothing controls.


In [23]:
# Inspect only tetrahedra below the chosen quality threshold.
SHOW_LOW_QUALITY_TETS = False
QUALITY_VIEW_THRESHOLD = 0.05
if SHOW_LOW_QUALITY_TETS:
    if verified_tets is None:
        raise RuntimeError("Run the tetrahedral verification cell first.")
    bad_ids = np.flatnonzero(verified_tets.quality < QUALITY_VIEW_THRESHOLD)
    print(f"{len(bad_ids):,} / {len(verified_tets.tetrahedra):,} below threshold")
    if len(bad_ids):
        _cells = np.column_stack((np.full(len(verified_tets.tetrahedra), 4),
                                  verified_tets.tetrahedra)).ravel()
        _grid = pv.UnstructuredGrid(_cells,
            np.full(len(verified_tets.tetrahedra), pv.CellType.TETRA, dtype=np.uint8),
            verified_tets.points)
        _grid.cell_data['minSICN'] = verified_tets.quality
        _grid.cell_data['grain_id'] = verified_tets.grain_ids
        _poor = _grid.extract_cells(bad_ids)
        _plot = pv.Plotter()
        _plot.add_mesh(_poor, scalars='minSICN', show_edges=True,
                       clim=(0, QUALITY_VIEW_THRESHOLD))
        _plot.add_axes()
        _plot.show()


## All interfaces, RVE caps and physical junctions

Uses `repaired_rve`. Surface edges are hidden. Internal and RVE cap opacities are independent. Junction points are branches/endpoints of the physical line network or nodes incident to four or more regions; this does not display every line discretization node. Shared interfaces are drawn once and coloured by one incident grain ID. These cells only visualize existing results.

In [24]:
from importlib import import_module, reload
_views = reload(import_module('upxo.meshing.gbconformant.d3v2p0.views'))
SHOW_RVE_FACES = True
INTERNAL_OPACITY = 0.35
RVE_OPACITY = 0.15
SURFACE_CMAP = 'nipy_spectral'
SHOW_JUNCTION_LINES = True
JUNCTION_LINE_COLOR = 'red'
JUNCTION_LINE_WIDTH = 3.0
SHOW_JUNCTION_POINTS = False
JUNCTION_POINT_COLOR = 'black'
JUNCTION_POINT_SIZE = 10.0
INCLUDE_RVE_JUNCTIONS = True  # Exterior counts as a region; includes GB traces on RVE faces.

_surface_view = _views.surface_view(
    repaired_rve, internal_grains_only=False,
    show_rve=SHOW_RVE_FACES, internal_opacity=INTERNAL_OPACITY,
    rve_opacity=RVE_OPACITY, cmap=SURFACE_CMAP,
    show_lines=SHOW_JUNCTION_LINES, line_color=JUNCTION_LINE_COLOR,
    line_width=JUNCTION_LINE_WIDTH, show_points=SHOW_JUNCTION_POINTS,
    point_color=JUNCTION_POINT_COLOR, point_size=JUNCTION_POINT_SIZE,
    include_rve_junctions=INCLUDE_RVE_JUNCTIONS)


Displaying boundaries incident to 373 grains


## Tetrahedra of mutually non-touching grains

Uses matching `repaired_rve` and `tet_grid` results. “Non-touching” here means selected grains share no mesh node (including point and edge contact), not “not touching the RVE”. A seeded greedy selection produces a maximal set unless `MAX_GRAINS` stops it early; it does not solve for the largest possible set. Segment IDs identify disconnected pieces by mesh connectivity and are visualization labels only. Changing the selection can renumber segments. Junction overlays show the physical junctions incident to selected grains in the full complex.

In [25]:
from importlib import import_module, reload
_views = reload(import_module('upxo.meshing.gbconformant.d3v2p0.views'))
SELECTION_SEED = 42  # Change to select another mutually non-touching set.
MAX_GRAINS = None  # Positive integer to limit the number displayed.
COLOR_BY_SEGMENT = True  # False: grain ID; True: disconnected-component segment ID.
TET_CMAP = 'nipy_spectral'
TET_OPACITY = 1.0
TET_SHOW_EDGES = False
SHOW_JUNCTION_LINES = True
JUNCTION_LINE_COLOR = 'red'
JUNCTION_LINE_WIDTH = 3.0
SHOW_JUNCTION_POINTS = False
JUNCTION_POINT_COLOR = 'black'
JUNCTION_POINT_SIZE = 10.0
INCLUDE_RVE_JUNCTIONS = True
CLIP_ENABLED = False
CLIP_CRINKLE = True  # Keep whole intersected tetrahedra.
CLIP_NORMAL = 'x'
CLIP_ORIGIN = None  # None uses the selected mesh centre; or specify (x, y, z).
CLIP_INVERT = False

_tet_view, displayed_grain_ids = _views.tet_view(
    repaired_rve, tet_grid, seed=SELECTION_SEED, maximum=MAX_GRAINS,
    color_segments=COLOR_BY_SEGMENT, cmap=TET_CMAP, opacity=TET_OPACITY,
    show_edges=TET_SHOW_EDGES,
    show_lines=SHOW_JUNCTION_LINES, line_color=JUNCTION_LINE_COLOR,
    line_width=JUNCTION_LINE_WIDTH, show_points=SHOW_JUNCTION_POINTS,
    point_color=JUNCTION_POINT_COLOR, point_size=JUNCTION_POINT_SIZE,
    include_rve_junctions=INCLUDE_RVE_JUNCTIONS,
    clip=CLIP_ENABLED, crinkle=CLIP_CRINKLE, normal=CLIP_NORMAL,
    origin=CLIP_ORIGIN, invert=CLIP_INVERT)


146 mutually non-touching grains; 862,900 tetrahedra
Selected grain IDs: [133, 135, 108, 204, 255, 157, 101, 358, 59, 372, 211, 128, 363, 175, 254, 226, 27, 141, 201, 216, 233, 57, 132, 144, 153, 316, 270, 81, 286, 302, 353, 199, 306, 184, 337, 365, 342, 83, 90, 215, 99, 179, 145, 69, 312, 30, 355, 338, 13, 193, 262, 346, 231, 85, 172, 323, 102, 74, 294, 194, 31, 243, 167, 214, 26, 277, 218, 356, 147, 328, 180, 151, 295, 162, 373, 217, 174, 113, 19, 154, 256, 308, 235, 80, 203, 332, 67, 10, 261, 60, 152, 359, 239, 257, 5, 271, 49, 68, 94, 95, 236, 84, 125, 369, 246, 345, 247, 252, 177, 362, 56, 263, 29, 12, 266, 40, 189, 14, 285, 171, 124, 309, 100, 361, 118, 117, 64, 224, 2, 43, 123, 18, 168, 298, 310, 241, 111, 334, 183, 50, 103, 140, 42, 304, 142, 37]


## Non-touching grains with an interactive crinkle clip plane

Same selection and styling as the previous view, with an interactive plane and whole-tetrahedron crinkle clipping enabled. Use the same seed and grain limit to reproduce the selection. Junction lines and points remain complete as spatial context; they are not clipped by the plane.

In [26]:
from importlib import import_module, reload
_views = reload(import_module('upxo.meshing.gbconformant.d3v2p0.views'))
SELECTION_SEED = 42  # Change to select another mutually non-touching set.
MAX_GRAINS = None  # Positive integer to limit the number displayed.
COLOR_BY_SEGMENT = True  # False: grain ID; True: disconnected-component segment ID.
TET_CMAP = 'nipy_spectral'
TET_OPACITY = 1.0
TET_SHOW_EDGES = False
SHOW_JUNCTION_LINES = True
JUNCTION_LINE_COLOR = 'red'
JUNCTION_LINE_WIDTH = 3.0
SHOW_JUNCTION_POINTS = False
JUNCTION_POINT_COLOR = 'black'
JUNCTION_POINT_SIZE = 10.0
INCLUDE_RVE_JUNCTIONS = True
CLIP_ENABLED = True
CLIP_CRINKLE = True  # Keep whole intersected tetrahedra.
CLIP_NORMAL = 'x'
CLIP_ORIGIN = None  # None uses the selected mesh centre; or specify (x, y, z).
CLIP_INVERT = False

_tet_view, displayed_grain_ids = _views.tet_view(
    repaired_rve, tet_grid, seed=SELECTION_SEED, maximum=MAX_GRAINS,
    color_segments=COLOR_BY_SEGMENT, cmap=TET_CMAP, opacity=TET_OPACITY,
    show_edges=TET_SHOW_EDGES,
    show_lines=SHOW_JUNCTION_LINES, line_color=JUNCTION_LINE_COLOR,
    line_width=JUNCTION_LINE_WIDTH, show_points=SHOW_JUNCTION_POINTS,
    point_color=JUNCTION_POINT_COLOR, point_size=JUNCTION_POINT_SIZE,
    include_rve_junctions=INCLUDE_RVE_JUNCTIONS,
    clip=CLIP_ENABLED, crinkle=CLIP_CRINKLE, normal=CLIP_NORMAL,
    origin=CLIP_ORIGIN, invert=CLIP_INVERT)


146 mutually non-touching grains; 862,900 tetrahedra
Selected grain IDs: [133, 135, 108, 204, 255, 157, 101, 358, 59, 372, 211, 128, 363, 175, 254, 226, 27, 141, 201, 216, 233, 57, 132, 144, 153, 316, 270, 81, 286, 302, 353, 199, 306, 184, 337, 365, 342, 83, 90, 215, 99, 179, 145, 69, 312, 30, 355, 338, 13, 193, 262, 346, 231, 85, 172, 323, 102, 74, 294, 194, 31, 243, 167, 214, 26, 277, 218, 356, 147, 328, 180, 151, 295, 162, 373, 217, 174, 113, 19, 154, 256, 308, 235, 80, 203, 332, 67, 10, 261, 60, 152, 359, 239, 257, 5, 271, 49, 68, 94, 95, 236, 84, 125, 369, 246, 345, 247, 252, 177, 362, 56, 263, 29, 12, 266, 40, 189, 14, 285, 171, 124, 309, 100, 361, 118, 117, 64, 224, 2, 43, 123, 18, 168, 298, 310, 241, 111, 334, 183, 50, 103, 140, 42, 304, 142, 37]


## Interfaces and junctions of grains wholly inside the RVE

Excludes every grain having any surface node on an RVE plane, including isolated point or edge contact. Displays the full boundaries of the remaining grains, including interfaces with excluded neighbours. No RVE caps are drawn in this view. Junction classification uses the full complex before filtering, so cropping does not invent new junctions.

In [ ]:
from importlib import import_module, reload
_views = reload(import_module('upxo.meshing.gbconformant.d3v2p0.views'))
SHOW_RVE_FACES = False
INTERNAL_OPACITY = 0.35
RVE_OPACITY = 1.0
SURFACE_CMAP = 'nipy_spectral'
SHOW_JUNCTION_LINES = True
JUNCTION_LINE_COLOR = 'red'
JUNCTION_LINE_WIDTH = 3.0
SHOW_JUNCTION_POINTS = False
JUNCTION_POINT_COLOR = 'black'
JUNCTION_POINT_SIZE = 10.0
INCLUDE_RVE_JUNCTIONS = True  # Exterior counts as a region; includes GB traces on RVE faces.

_surface_view = _views.surface_view(
    repaired_rve, internal_grains_only=True,
    show_rve=SHOW_RVE_FACES, internal_opacity=INTERNAL_OPACITY,
    rve_opacity=RVE_OPACITY, cmap=SURFACE_CMAP,
    show_lines=SHOW_JUNCTION_LINES, line_color=JUNCTION_LINE_COLOR,
    line_width=JUNCTION_LINE_WIDTH, show_points=SHOW_JUNCTION_POINTS,
    point_color=JUNCTION_POINT_COLOR, point_size=JUNCTION_POINT_SIZE,
    include_rve_junctions=INCLUDE_RVE_JUNCTIONS)


Displaying boundaries incident to 193 grains


## Linked voxel / surface / junction comparison and tet-quality distribution

Left to right: voxellated grains, internal surface mesh, transparent internal surfaces with physical junction lines, and a minSICN histogram of **all tetrahedra**. Rotate, pan or zoom any of the first three panels to move their shared camera; the histogram remains independent. The opaque voxel view shows the outside of the labelled voxel block; reduce `VOXEL_OPACITY` to see through it.

The default voxel source is the cleaned dataset before refinement (`labels_meshing`, with `SPACING`). To inspect the refined voxel grid, use `labels_upscaled` with `effective_spacing`. All geometry must come from the same pipeline run. The colour scale is shared across the geometry panels; each internal interface is drawn once and coloured by its first incident grain. RVE caps are optional on the surface panels. This cell only visualizes existing results and does not rerun meshing.


In [60]:
from importlib import import_module, reload
_views = reload(import_module('upxo.meshing.gbconformant.d3v2p0.views'))

COMPARISON_LABELS = labels_meshing
COMPARISON_SPACING = SPACING
COMPARISON_CMAP = 'nipy_spectral'
VOXEL_OPACITY = 1.0
SURFACE_OPACITY = 1.0
TRANSPARENT_SURFACE_OPACITY = 0.2
SHOW_RVE_CAPS = False
RVE_CAP_OPACITY = 0.1
JUNCTION_LINE_COLOR = 'red'
JUNCTION_LINE_WIDTH = 3.0
SHOW_JUNCTION_POINTS = False
JUNCTION_POINT_COLOR = 'black'
JUNCTION_POINT_SIZE = 10.0
INCLUDE_RVE_JUNCTIONS = True
QUALITY_HISTOGRAM_BINS = 60
QUALITY_LOG_COUNTS = False
QUALITY_TARGET = MIN_TET_QUALITY
COMPARISON_WINDOW_SIZE = (2000, 650)

if verified_tets is None:
    raise RuntimeError('Run the tetrahedral verification cell first.')
linked_comparison_plotter = _views.linked_comparison(
    COMPARISON_LABELS, COMPARISON_SPACING, repaired_rve, verified_tets.quality,
    cmap=COMPARISON_CMAP, voxel_opacity=VOXEL_OPACITY,
    surface_opacity=SURFACE_OPACITY, transparent_opacity=TRANSPARENT_SURFACE_OPACITY,
    show_rve=SHOW_RVE_CAPS, rve_opacity=RVE_CAP_OPACITY,
    line_color=JUNCTION_LINE_COLOR, line_width=JUNCTION_LINE_WIDTH,
    show_points=SHOW_JUNCTION_POINTS, point_color=JUNCTION_POINT_COLOR,
    point_size=JUNCTION_POINT_SIZE, include_rve_junctions=INCLUDE_RVE_JUNCTIONS,
    bins=QUALITY_HISTOGRAM_BINS, log_counts=QUALITY_LOG_COUNTS,
    quality_threshold=QUALITY_TARGET, window_size=COMPARISON_WINDOW_SIZE)


## Interactive grain-ID selector: voxels / tetrahedra / boundary segments

Select a grain using the slider in the left panel; release it to update all three panels. **Right/Up** selects the next available ID; **Left/Down** selects the previous ID. The slider steps through existing IDs, so gaps in numbering are supported. Its title displays the actual grain ID. Set `INITIAL_GRAIN_ID` for a specific starting grain. All three cameras are linked. `RESET_CAMERA_ON_SELECTION` fits each newly selected grain; disable it to retain the current camera.

Left: cleaned voxellated grain. Middle: its tetrahedra with physical junction lines. Right: boundary triangles coloured by **surface-segment ID**, with junction lines. A segment is an edge-connected patch with the same neighbouring grain and RVE-face ownership; disconnected patches receive separate IDs. These IDs are local to the selected grain and are independent of artificial Gmsh charts. RVE caps can be hidden. Junction overlays show where the selected grain meets the full grain complex, including RVE traces if enabled.

This reads matching `labels_meshing`, `repaired_rve` and `tet_grid` from the current run. Membership indexes and physical junctions are computed once when the viewer starts. No meshing is rerun. Set `INITIAL_GRAIN_ID` and rerun this visualization cell to start at an exact ID; live selection uses the slider or arrow keys.


In [32]:
from importlib import import_module, reload
_views = reload(import_module('upxo.meshing.gbconformant.d3v2p0.views'))

INITIAL_GRAIN_ID = None  # None starts at the smallest existing grain ID.
SELECTOR_LABELS = labels_meshing
SELECTOR_SPACING = SPACING
GRAIN_CMAP = 'nipy_spectral'
SEGMENT_CMAP = 'tab20'
VOXEL_OPACITY = 1.0
TET_OPACITY = 1.0
BOUNDARY_OPACITY = 1.0
VOXEL_SHOW_EDGES = True
TET_SHOW_EDGES = False
BOUNDARY_SHOW_EDGES = True
SHOW_RVE_CAPS = True
SHOW_JUNCTION_LINES = True
JUNCTION_LINE_COLOR = 'red'
JUNCTION_LINE_WIDTH = 3.0
SHOW_JUNCTION_POINTS = False
JUNCTION_POINT_COLOR = 'black'
JUNCTION_POINT_SIZE = 10.0
INCLUDE_RVE_JUNCTIONS = True
RESET_CAMERA_ON_SELECTION = True
SELECTOR_WINDOW_SIZE = (1800, 700)

grain_selector_plotter, select_grain_id = _views.grain_selector(
    SELECTOR_LABELS, SELECTOR_SPACING, repaired_rve, tet_grid,
    initial_grain=INITIAL_GRAIN_ID, grain_cmap=GRAIN_CMAP, segment_cmap=SEGMENT_CMAP,
    voxel_opacity=VOXEL_OPACITY, tet_opacity=TET_OPACITY, surface_opacity=BOUNDARY_OPACITY,
    voxel_edges=VOXEL_SHOW_EDGES, tet_edges=TET_SHOW_EDGES, surface_edges=BOUNDARY_SHOW_EDGES, show_rve=SHOW_RVE_CAPS,
    show_lines=SHOW_JUNCTION_LINES, line_color=JUNCTION_LINE_COLOR,
    line_width=JUNCTION_LINE_WIDTH, show_points=SHOW_JUNCTION_POINTS,
    point_color=JUNCTION_POINT_COLOR, point_size=JUNCTION_POINT_SIZE,
    include_rve_junctions=INCLUDE_RVE_JUNCTIONS,
    reset_camera=RESET_CAMERA_ON_SELECTION, window_size=SELECTOR_WINDOW_SIZE)


## Linked grain selector with boundary curvature

Left: voxellated grain with edges enabled. Middle: the selected tetrahedral mesh's **outer boundary**, coloured by curvature. Right: grain-boundary triangles coloured by the same curvature values. Both surface views include configurable junction lines and points. Curvature is a surface quantity, not a tetrahedral volume-element quantity.

`CURVATURE_TYPE` accepts `'mean'` (signed, inverse-length units) or `'gaussian'` (inverse-length-squared units). It is estimated separately on each grain's complete, outward-oriented closed triangular boundary, including RVE caps. This avoids calculating curvature across the globally nonmanifold grain complex. Junctions, sharp cap edges and irregular triangles can produce large discrete-curvature values; these are diagnostic estimates, not the planned multiscale curvature-preservation method. Opposite sides of a shared interface can have different signed mean curvature.

The middle and right panels use identical values at matching boundary nodes. With `CURVATURE_LIMITS = None`, the viewer computes the minimum and maximum over **all grain boundaries once at startup**, then keeps that range fixed during selection. Alternatively specify a fixed `(minimum, maximum)` tuple; values outside it saturate at the end colours. `CURVATURE_CMAP` defaults to `'tab20c'`. No smoothing or remeshing is performed. Startup requires one curvature calculation per grain, cached for subsequent selections.

Use the slider or arrow keys as in the preceding selector. Set `INITIAL_GRAIN_ID` for an exact starting ID. The cameras are linked across all three views.


In [36]:
from importlib import import_module, reload
_views = reload(import_module('upxo.meshing.gbconformant.d3v2p0.views'))

INITIAL_GRAIN_ID = None  # None starts at the smallest existing grain ID.
SELECTOR_LABELS = labels_meshing
SELECTOR_SPACING = SPACING
GRAIN_CMAP = 'nipy_spectral'
SEGMENT_CMAP = 'tab20'  # Unused when curvature colouring is enabled.
CURVATURE_TYPE = 'mean'
CURVATURE_CMAP = 'tab20c'
CURVATURE_LIMITS = None  # Global fixed range, or specify e.g. (-2.0, 2.0).
VOXEL_OPACITY = 1.0
TET_OPACITY = 1.0
BOUNDARY_OPACITY = 1.0
VOXEL_SHOW_EDGES = True
TET_SHOW_EDGES = False
BOUNDARY_SHOW_EDGES = True
SHOW_RVE_CAPS = True
SHOW_JUNCTION_LINES = True
JUNCTION_LINE_COLOR = 'red'
JUNCTION_LINE_WIDTH = 3.0
SHOW_JUNCTION_POINTS = False
JUNCTION_POINT_COLOR = 'black'
JUNCTION_POINT_SIZE = 10.0
INCLUDE_RVE_JUNCTIONS = True
RESET_CAMERA_ON_SELECTION = True
SELECTOR_WINDOW_SIZE = (1800, 700)

curvature_selector_plotter, select_curvature_grain_id = _views.grain_selector(
    SELECTOR_LABELS, SELECTOR_SPACING, repaired_rve, tet_grid,
    initial_grain=INITIAL_GRAIN_ID, grain_cmap=GRAIN_CMAP, segment_cmap=SEGMENT_CMAP,
    voxel_opacity=VOXEL_OPACITY, tet_opacity=TET_OPACITY, surface_opacity=BOUNDARY_OPACITY,
    voxel_edges=VOXEL_SHOW_EDGES, tet_edges=TET_SHOW_EDGES, surface_edges=BOUNDARY_SHOW_EDGES, show_rve=SHOW_RVE_CAPS,
    show_lines=SHOW_JUNCTION_LINES, line_color=JUNCTION_LINE_COLOR,
    line_width=JUNCTION_LINE_WIDTH, show_points=SHOW_JUNCTION_POINTS,
    point_color=JUNCTION_POINT_COLOR, point_size=JUNCTION_POINT_SIZE,
    include_rve_junctions=INCLUDE_RVE_JUNCTIONS,
    reset_camera=RESET_CAMERA_ON_SELECTION, window_size=SELECTOR_WINDOW_SIZE,
    curvature_type=CURVATURE_TYPE, curvature_cmap=CURVATURE_CMAP,
    curvature_limits=CURVATURE_LIMITS)


Computing per-grain boundary curvature and one fixed colour range...
Fixed curvature limits: (-16.39166057837835, 24.751941774569254)


## Boundary displacement map: original versus moved interfaces

Linked views show original voxel-interface geometry on the left and moved geometry on the right, coloured by the **same exact node displacement**. Displacement is measured from `original_points`, in the physical units of `SPACING`; it is not distance to the nearest triangle. Split pinch nodes retain their original parent-node reference.

`DISPLACEMENT_STAGE = 'pre_gmsh'` includes smoothing, geometry rollback and pinch separation. Choose `'smoothed'` for smoothing only, or `'guarded'` for smoothing plus geometry rollback. This does **not** measure later Gmsh remeshing or post-remeshing repair; those stages do not retain a general one-to-one node correspondence. Voxel-label changes during cleaning and newly generated RVE caps are also outside this map.

Set `DISPLACEMENT_GRAIN_ID` to a grain ID or `None` for all interfaces. `DISPLACEMENT_COMPONENT` can be `'magnitude'`, `'x'`, `'y'` or `'z'`. Automatic colour limits use all nodes of the selected stage, so changing the grain filter does not rescale the map. A fixed tuple overrides these limits. Magnitudes default to a nonnegative range; signed components use a symmetric range. Junction lines and frozen junction points are optional. The geometry itself is never changed by this cell.


In [37]:
from importlib import import_module, reload
_views = reload(import_module('upxo.meshing.gbconformant.d3v2p0.views'))

DISPLACEMENT_STAGE = 'pre_gmsh'  # 'smoothed', 'guarded', or 'pre_gmsh'
DISPLACEMENT_GRAIN_ID = None  # None: all internal interfaces.
DISPLACEMENT_COMPONENT = 'magnitude'  # Or 'x', 'y', 'z'.
DISPLACEMENT_CMAP = 'viridis'  # E.g. 'tab20c'; 'coolwarm' suits signed components.
DISPLACEMENT_LIMITS = None  # Global range for this stage, or a fixed (min, max).
DISPLACEMENT_OPACITY = 1.0
DISPLACEMENT_SHOW_EDGES = False
SHOW_JUNCTION_LINES = True
JUNCTION_LINE_COLOR = 'red'
JUNCTION_LINE_WIDTH = 3.0
SHOW_JUNCTION_POINTS = False
JUNCTION_POINT_COLOR = 'black'
JUNCTION_POINT_SIZE = 10.0
DISPLACEMENT_WINDOW_SIZE = (1500, 700)

_displacement_sources = {
    'smoothed': interfaces_smoothed,
    'guarded': interfaces_before_unpinch,
    'pre_gmsh': interfaces,
}
if DISPLACEMENT_STAGE not in _displacement_sources:
    raise ValueError('Select smoothed, guarded, or pre_gmsh.')
displacement_plotter = _views.displacement_view(
    _displacement_sources[DISPLACEMENT_STAGE], grain_id=DISPLACEMENT_GRAIN_ID,
    component=DISPLACEMENT_COMPONENT, cmap=DISPLACEMENT_CMAP,
    limits=DISPLACEMENT_LIMITS, opacity=DISPLACEMENT_OPACITY,
    show_edges=DISPLACEMENT_SHOW_EDGES,
    show_lines=SHOW_JUNCTION_LINES, line_color=JUNCTION_LINE_COLOR,
    line_width=JUNCTION_LINE_WIDTH, show_points=SHOW_JUNCTION_POINTS,
    point_color=JUNCTION_POINT_COLOR, point_size=JUNCTION_POINT_SIZE,
    window_size=DISPLACEMENT_WINDOW_SIZE)


Exact node displacements, in SPACING physical units.
Selected nodes: 271,591; mean magnitude: 0.389825; maximum magnitude: 2.00427; colour limits: (0.0, 2.004272735606207)


## Per-grain volume-change map

Linked views show reference voxel geometry on the left and final closed grain surfaces on the right, coloured by per-grain volume change. **Negative means shrinkage; positive means expansion.** Percentage change is `100 * (final - reference) / reference`. Absolute change has physical volume units (the cube of the length unit in `SPACING`).

`VOLUME_BASELINE = 'cleaned'` compares final geometry against `labels_meshing` before upscaling, isolating subsequent smoothing and repair/remeshing changes. Choose `'imported'` to include the effect of voxel cleaning as well; this uses the already shuffled `labels`, not the unshuffled `labels_original`. The final volumes are recomputed from the oriented closed triangles of `repaired_rve`, including all RVE caps, regardless of their display visibility. This is a diagnostic, not an implementation of volume preservation or a new topology certification.

The automatic colour range is symmetric about zero and uses all grain values, so changing `VOLUME_GRAIN_ID` does not change the range. Set fixed `VOLUME_LIMITS` to focus on smaller changes; out-of-range values saturate at the colour endpoints. Tiny grains can show very large percentages, so the printed table and returned `grain_volume_change_data` contain both absolute and percentage changes. The table ranks the largest absolute percentage changes.

For all-grain display, each shared interface is drawn once and coloured by its first incident grain's change; the two grains can have different changes. Select `VOLUME_GRAIN_ID` to see that grain's entire boundary coloured consistently by its own value. Internal and RVE-face opacities are independent. No geometry is modified.


In [39]:
from importlib import import_module, reload
_views = reload(import_module('upxo.meshing.gbconformant.d3v2p0.views'))

VOLUME_BASELINE = 'cleaned'  # 'cleaned' or 'imported'
VOLUME_GRAIN_ID = None  # None: all grains; or one shuffled grain ID.
VOLUME_CHANGE_MODE = 'percent'  # Or 'absolute'.
VOLUME_CMAP = 'nipy_spectral'
VOLUME_LIMITS = None  # Automatic symmetric global range, or e.g. (-20.0, 20.0).
VOLUME_VOXEL_EDGES = True
VOLUME_VOXEL_OPACITY = 1.0
VOLUME_SHOW_INTERNAL = True
VOLUME_SHOW_RVE_FACES = True
VOLUME_INTERNAL_OPACITY = 0.35
VOLUME_RVE_OPACITY = 0.35
VOLUME_SURFACE_EDGES = False
SHOW_JUNCTION_LINES = False
JUNCTION_LINE_COLOR = 'red'
JUNCTION_LINE_WIDTH = 3.0
SHOW_JUNCTION_POINTS = False
JUNCTION_POINT_COLOR = 'black'
JUNCTION_POINT_SIZE = 10.0
VOLUME_TABLE_TOP_N = 20
VOLUME_WINDOW_SIZE = (1500, 700)

if VOLUME_BASELINE not in ('cleaned', 'imported'):
    raise ValueError('Select cleaned or imported for VOLUME_BASELINE.')
volume_change_plotter, grain_volume_change_data = _views.volume_change_view(
    labels_meshing if VOLUME_BASELINE == 'cleaned' else labels,
    SPACING, repaired_rve, grain_id=VOLUME_GRAIN_ID, mode=VOLUME_CHANGE_MODE,
    cmap=VOLUME_CMAP, limits=VOLUME_LIMITS,
    voxel_edges=VOLUME_VOXEL_EDGES, voxel_opacity=VOLUME_VOXEL_OPACITY,
    internal_opacity=VOLUME_INTERNAL_OPACITY, rve_opacity=VOLUME_RVE_OPACITY,
    show_internal=VOLUME_SHOW_INTERNAL, show_rve=VOLUME_SHOW_RVE_FACES,
    show_edges=VOLUME_SURFACE_EDGES,
    show_lines=SHOW_JUNCTION_LINES, line_color=JUNCTION_LINE_COLOR,
    line_width=JUNCTION_LINE_WIDTH, show_points=SHOW_JUNCTION_POINTS,
    point_color=JUNCTION_POINT_COLOR, point_size=JUNCTION_POINT_SIZE,
    top_n=VOLUME_TABLE_TOP_N, window_size=VOLUME_WINDOW_SIZE)


Negative = shrinkage; positive = expansion. Volumes in physical units cubed.
Reference total: 125000; final total: 125000
Grain ID    Reference volume      Final volume     Absolute change      Change (%)
      67                   1      0.0015068456         -0.99849315        -99.8493
     235                   1      0.0015084082         -0.99849159        -99.8492
      10                   1      0.0015091409         -0.99849086        -99.8491
      61                   1      0.0015096027          -0.9984904         -99.849
      13                   1      0.0015096101         -0.99849039         -99.849
     173                   1      0.0015096101         -0.99849039         -99.849
     304                   1      0.0015122571         -0.99848774        -99.8488
     244                   1      0.0015134327         -0.99848657        -99.8487
       2                   1      0.0015187498         -0.99848125        -99.8481
     335                   1      0.0015187509  

## Sampled thickness / boundary-separation map

Choose a grain ID. Left: distance from a sampled triangle centroid **inward**, along its normal, to the next surface of that same grain. Right: distance **outward** to the next surface in the full grain complex. The right quantity is spacing between boundary sheets through the neighbouring material—not an empty gap between conforming grains, which share their interface. Outward measurements from RVE caps are undefined.

**These are directional sampled diagnostics, not rigorous minimum thickness or minimum separation guards.** Oblique thin features can be missed, and the reported minimum is only the smallest sampled normal-ray distance. This does not implement the deferred geometry guards. Each coloured triangle represents one centroid measurement; grey triangles are unsampled, undefined, or have no ray hit. No interpolation fills missing measurements. The output distinguishes undefined RVE-cap rays from no-hit rays.

`THICKNESS_MAX_SAMPLES` bounds startup cost; increasing it gives greater triangle coverage. Sampling is reproducible and uniform over triangles, not area-weighted. `RAY_OFFSET` avoids hitting the starting triangle; `None` derives a small offset from RVE size. Features below that offset cannot be resolved reliably. `MAX_RAY_DISTANCE` can limit the search; a no-hit result is NaN, never zero. Spatial locators are built once for each run of this cell. No smoothing, cleaning or meshing is executed.

Colour limits default to `(0, RVE diagonal)`, shared by both panels and unchanged when selecting another grain. Set a narrower fixed `THICKNESS_LIMITS` for more contrast, e.g. `(0, 3)` in your physical length units. Cmap, opacity, triangle edges and junction overlays are configurable. Measurements and their source triangle IDs are returned in `thickness_separation_data`.


In [ ]:
from importlib import import_module, reload
_thickness = reload(import_module('upxo.meshing.gbconformant.d3v2p0.thickness_view'))

THICKNESS_GRAIN_ID = int(np.unique(repaired_rve.grain_pairs)[2])  # Set an existing shuffled ID.
THICKNESS_MAX_SAMPLES = 15000
THICKNESS_SAMPLE_SEED = 42
RAY_OFFSET = None
MAX_RAY_DISTANCE = None
THICKNESS_CMAP = 'nipy_spectral'  # E.g. 'tab20c'.
# THICKNESS_LIMITS = None  # Fixed RVE-based range, or e.g. (0.0, 3.0).
THICKNESS_LIMITS = (0, 25)  # Fixed RVE-based range, or e.g. (0.0, 3.0).
THICKNESS_CONTEXT_OPACITY = 0.12
THICKNESS_MAP_OPACITY = 1.0
THICKNESS_SHOW_EDGES = False
SHOW_JUNCTION_LINES = True
JUNCTION_LINE_COLOR = 'red'
JUNCTION_LINE_WIDTH = 3.0
SHOW_JUNCTION_POINTS = True
JUNCTION_POINT_COLOR = 'black'
JUNCTION_POINT_SIZE = 10.0
THICKNESS_WINDOW_SIZE = (1500, 700)

thickness_separation_plotter, thickness_separation_data = _thickness.thickness_separation_view(
    repaired_rve, THICKNESS_GRAIN_ID,
    max_samples=THICKNESS_MAX_SAMPLES, seed=THICKNESS_SAMPLE_SEED,
    ray_offset=RAY_OFFSET, max_distance=MAX_RAY_DISTANCE,
    cmap=THICKNESS_CMAP, limits=THICKNESS_LIMITS,
    context_opacity=THICKNESS_CONTEXT_OPACITY, map_opacity=THICKNESS_MAP_OPACITY,
    show_edges=THICKNESS_SHOW_EDGES,
    show_lines=SHOW_JUNCTION_LINES, line_color=JUNCTION_LINE_COLOR,
    line_width=JUNCTION_LINE_WIDTH, show_points=SHOW_JUNCTION_POINTS,
    point_color=JUNCTION_POINT_COLOR, point_size=JUNCTION_POINT_SIZE,
    window_size=THICKNESS_WINDOW_SIZE)


Building spatial indexes; tracing 11,760 sampled faces for grain 3...
{'grain_id': 3, 'total_grain_triangles': 11760, 'sampled_triangles': 11760, 'valid_thickness': 11760, 'valid_separation': 11760, 'undefined_outward_rve_caps': 0, 'thickness_no_hit': 0, 'separation_no_hit': 0, 'ray_offset': 8.660254037844386e-08, 'numerical_tolerance': 8.660254037844386e-09, 'max_ray_distance': 173.20508075688772, 'minimum_thickness_certified': False, 'minimum_separation_certified': False, 'sampling': 'seeded uniform triangle sampling; not area-weighted'}
thickness {'minimum_sampled': 0.039870666103432435, 'median_sampled': 4.116517812394227, 'maximum_sampled': 22.957698994139843}
separation {'minimum_sampled': 0.03430543629138907, 'median_sampled': 4.025346627185684, 'maximum_sampled': 29.104942444241917}


## Quality histogram linked to tetrahedral selection

The left panel displays whole tetrahedra whose **minSICN falls inside the inclusive lower/upper range**. Release either slider to update the mesh, selected count, and orange histogram bars on the right. Grey bars retain the full eligible distribution; black lines show the active interval and the red dashed line marks the quality target. The threshold controls visualization only, not mesh acceptance. Bins, histogram axes and the mesh colour scale stay fixed while selecting. With log counts, zero-count bins are simply absent.

`QUALITY_FILTER_GRAIN_ID = None` uses all tetrahedra; specify a grain ID to restrict both the histogram and selection. The initial interval defaults to `[0, MIN_TET_QUALITY]` to locate low-quality elements (including values equal to the target). Lower/upper controls cannot cross. A zero-result selection clears the highlighted mesh and reports zero explicitly. Mesh extraction occurs only on slider release; a broad interval may select millions of tetrahedra and take longer to redraw.

Context displays the RVE caps for all-grain selection, or the selected grain's boundary when filtered. Junction overlays are optional. `quality_selection_state['cell_ids']` contains the **original `tet_grid` cell IDs** for the current selection. This cell reads existing tetrahedra and does not remesh, optimize or modify their qualities.


In [52]:
from importlib import import_module, reload
_quality_view = reload(import_module('upxo.meshing.gbconformant.d3v2p0.quality_view'))

QUALITY_FILTER_GRAIN_ID = None
QUALITY_INITIAL_RANGE = (0.0, MIN_TET_QUALITY)
QUALITY_HISTOGRAM_BINS = 80
QUALITY_HISTOGRAM_LOG_COUNTS = True
QUALITY_TARGET_LINE = MIN_TET_QUALITY
QUALITY_MESH_CMAP = 'viridis'  # E.g. 'tab20c'.
QUALITY_MESH_COLOR_LIMITS = (0.0, 1.0)
QUALITY_SELECTED_OPACITY = 1.0
QUALITY_SELECTED_SHOW_EDGES = True
QUALITY_SHOW_CONTEXT = True
QUALITY_CONTEXT_OPACITY = 0.08
SHOW_JUNCTION_LINES = False
JUNCTION_LINE_COLOR = 'red'
JUNCTION_LINE_WIDTH = 3.0
SHOW_JUNCTION_POINTS = False
JUNCTION_POINT_COLOR = 'black'
JUNCTION_POINT_SIZE = 10.0
QUALITY_WINDOW_SIZE = (1600, 750)

quality_histogram_plotter, quality_selection_state, select_quality_range = _quality_view.quality_histogram_view(
    tet_grid, repaired_rve, grain_id=QUALITY_FILTER_GRAIN_ID,
    initial_range=QUALITY_INITIAL_RANGE, bins=QUALITY_HISTOGRAM_BINS,
    cmap=QUALITY_MESH_CMAP, color_limits=QUALITY_MESH_COLOR_LIMITS,
    selected_opacity=QUALITY_SELECTED_OPACITY, show_edges=QUALITY_SELECTED_SHOW_EDGES,
    show_context=QUALITY_SHOW_CONTEXT, context_opacity=QUALITY_CONTEXT_OPACITY,
    log_counts=QUALITY_HISTOGRAM_LOG_COUNTS, target=QUALITY_TARGET_LINE,
    show_lines=SHOW_JUNCTION_LINES, line_color=JUNCTION_LINE_COLOR,
    line_width=JUNCTION_LINE_WIDTH, show_points=SHOW_JUNCTION_POINTS,
    point_color=JUNCTION_POINT_COLOR, point_size=JUNCTION_POINT_SIZE,
    window_size=QUALITY_WINDOW_SIZE)


## Defect-only view: saved surface diagnostics and current element quality

Uses the matching `repaired_rve` / `repaired_validation` pair, plus `tet_grid` when available. **Run the validation cell again if you changed the surface after it was checked**: older reports contain triangle indices and cannot establish that a modified surface is valid. This viewer does not repeat the full intersection or topology scan. For an earlier stage, substitute its surface and its matching validation report together, and disable tetrahedra unless they correspond to that stage.

Toggle categories below. Red: reported intersections; magenta: narrow facet openings; orange: poor surface-triangle quality; blue: remaining saved flags. The saved `bad_triangle_ids` mixes topology and quality reasons, so blue is **not** labelled as confirmed topology damage. A face with several flags is drawn once, with precedence red → magenta → orange → blue. Raw category counts can overlap. Gold tetrahedra are quality warnings, while black denotes nonpositive or nonfinite quality. These surface and tet quality metrics are different.

Coincident-node locations are recovered only when the report records such pairs, using its numerical tolerance. This is not a new configurable physical clearance guard. Unchecked diagnostics are printed explicitly; an empty plot does not certify absence of defects. Missing-grain and other global blockers may have no triangles to draw, so read the printed report as well.

Context and junction lines are off by default to keep the view defect-only. Grain filtering, colours, opacity and edges are configurable. The returned `defect_view_data` contains original triangle/node/tet indices. No repair, smoothing or meshing occurs.


In [53]:
from importlib import import_module, reload
_defect_view = reload(import_module('upxo.meshing.gbconformant.d3v2p0.defect_view'))

DEFECT_GRAIN_ID = None
SHOW_REPORTED_SURFACE_FLAGS = True
SHOW_INTERSECTIONS = True
SHOW_NARROW_FACET_ANGLES = True
SHOW_POOR_SURFACE_TRIANGLES = True
SHOW_COINCIDENT_NODES = True
SHOW_POOR_TETRAHEDRA = True
DEFECT_SURFACE_QUALITY_THRESHOLD = MIN_SURFACE_TRIANGLE_QUALITY
DEFECT_TET_QUALITY_THRESHOLD = MIN_TET_QUALITY
DEFECT_SHOW_CONTEXT = False
DEFECT_CONTEXT_OPACITY = 0.05
DEFECT_SURFACE_OPACITY = 1.0
DEFECT_TET_OPACITY = 1.0
DEFECT_SHOW_EDGES = True
DEFECT_POINT_SIZE = 12.0
DEFECT_COLORS = {
    'reported': 'royalblue', 'surface_quality': 'orange',
    'small_angles': 'magenta', 'intersections': 'red',
    'coincident_nodes': 'cyan', 'tet_quality': 'gold',
    'invalid_tet_quality': 'black',
}
SHOW_JUNCTION_LINES = False
JUNCTION_LINE_COLOR = 'red'
JUNCTION_LINE_WIDTH = 3.0

defect_plotter, defect_view_data = _defect_view.defect_only_view(
    repaired_rve, repaired_validation, globals().get('tet_grid') if SHOW_POOR_TETRAHEDRA else None,
    grain_id=DEFECT_GRAIN_ID, show_reported=SHOW_REPORTED_SURFACE_FLAGS,
    show_intersections=SHOW_INTERSECTIONS, show_small_angles=SHOW_NARROW_FACET_ANGLES,
    show_surface_quality=SHOW_POOR_SURFACE_TRIANGLES,
    show_coincident_nodes=SHOW_COINCIDENT_NODES, show_tet_quality=SHOW_POOR_TETRAHEDRA,
    surface_quality_threshold=DEFECT_SURFACE_QUALITY_THRESHOLD,
    tet_quality_threshold=DEFECT_TET_QUALITY_THRESHOLD,
    show_context=DEFECT_SHOW_CONTEXT, context_opacity=DEFECT_CONTEXT_OPACITY,
    surface_opacity=DEFECT_SURFACE_OPACITY, tet_opacity=DEFECT_TET_OPACITY,
    show_edges=DEFECT_SHOW_EDGES, point_size=DEFECT_POINT_SIZE, colors=DEFECT_COLORS,
    show_lines=SHOW_JUNCTION_LINES, line_color=JUNCTION_LINE_COLOR,
    line_width=JUNCTION_LINE_WIDTH)


Saved validation: PRELIMINARY_CHECKS_PASSED
Counts before display precedence; categories can overlap: {'reported': 3, 'surface_quality': 3, 'small_angles': 0, 'intersections': 0, 'coincident_nodes_displayed': 0, 'tet_quality': 5141, 'invalid_tet_quality': 0}
BLOCKERS: []
WARNINGS: ['3 triangles below quality threshold 0.05', 'Grain 153: 2 separate shells; containment needs checking', 'Grain 184: 2 separate shells; containment needs checking', 'Grain 185: 2 separate shells; containment needs checking', 'Grain 204: 2 separate shells; containment needs checking', 'Grain 279: 3 separate shells; containment needs checking', 'Grain 293: 2 separate shells; containment needs checking', 'Grain 301: 4 separate shells; containment needs checking', 'Grain 330: 2 separate shells; containment needs checking', 'Grain 343: 2 separate shells; containment needs checking']
UNCHECKED: ['minimum separation between nonadjacent surfaces', 'nested-shell containment and tetrahedron quality']
Dedicated physical

## Nested-island view

Displays a selected host grain's outer envelope transparently and enclosed grain envelopes inside it, including further nesting. `HOST_GRAIN_ID = None` automatically selects a host when nesting is detected. Host opacity applies to the selected host and intermediate hosts containing deeper islands; set it to zero to hide those envelopes. Colour by grain ID or relative nesting depth. Disconnected components are reported separately in `island_hierarchy`; shared host-cavity surfaces are not drawn twice.

Requires the previously validated, nonintersecting closed surfaces. Containment tests use all child-envelope vertices after bounding-box filtering, with floating-point VTK predicates. They are a visualization classification, not a replacement for intersection/containment certification. Only positive outer envelopes establish hierarchy; negative cavity shells are excluded from rendering. Detection may take time for many complex envelopes. An empty result is reported explicitly.

In [ ]:
from importlib import import_module, reload
_explore = reload(import_module('upxo.meshing.gbconformant.d3v2p0.exploration_views'))

HOST_GRAIN_ID = None
HOST_OPACITY = 0.08
ISLAND_OPACITY = 1.0
ISLAND_CMAP = 'tab20c'
ISLAND_COLOR_BY = 'grain_id'  # Or 'depth'.
ISLAND_SHOW_EDGES = False
ISLAND_MAX_DEPTH = None  # None: all levels; 1: host and immediate islands.

island_plotter, island_hierarchy = _explore.nested_island_view(
    repaired_rve, host_grain_id=HOST_GRAIN_ID, host_opacity=HOST_OPACITY,
    island_opacity=ISLAND_OPACITY, cmap=ISLAND_CMAP, color_by=ISLAND_COLOR_BY,
    show_edges=ISLAND_SHOW_EDGES, max_depth=ISLAND_MAX_DEPTH)


## Orthogonal slice viewer

Three axis-aligned sections controlled by X/Y/Z sliders. Release a slider to update that section. `SLICE_SOURCE` selects `'voxels'`, `'tets'` or `'both'`; both uses linked cameras, shared physical slice positions and the same grain-ID colour scale. These are planar sections, not crinkle cuts: intersected tetrahedra are cut geometrically.

Initial positions default to the RVE centre. Slider endpoints lie slightly inside the RVE to avoid ambiguous boundary-only intersections. The voxel view uses cleaned labels at original spacing; change the labels/spacing pair together to use refined voxels. Tet slices through millions of cells may take time to update; only the moved axis is recut.

In [ ]:
from importlib import import_module, reload
_explore = reload(import_module('upxo.meshing.gbconformant.d3v2p0.exploration_views'))

SLICE_SOURCE = 'both'  # 'voxels', 'tets', or 'both'.
SLICE_LABELS = labels_meshing
SLICE_SPACING = SPACING
SLICE_POSITIONS = None  # Physical (x, y, z), or None for the RVE centre.
SLICE_CMAP = 'tab20c'
SLICE_OPACITY = 1.0
SLICE_SHOW_EDGES = False
SLICE_WINDOW_SIZE = (1600, 750)

slice_plotter, slice_state, set_slice_position = _explore.orthogonal_slice_view(
    SLICE_LABELS, SLICE_SPACING, globals().get('tet_grid'), source=SLICE_SOURCE,
    positions=SLICE_POSITIONS, cmap=SLICE_CMAP, opacity=SLICE_OPACITY,
    show_edges=SLICE_SHOW_EDGES, window_size=SLICE_WINDOW_SIZE)


## Exploded grain view

Displays each grain's closed triangular shell, translated away from the RVE centre. Move the explosion slider to change separation; zero restores assembled positions. Coordinates and connectivity of `repaired_rve` and `tet_grid` remain untouched. This renders grain surfaces rather than duplicating all volume elements.

Translation is `factor * (grain surface-vertex mean centre - RVE centre)`. This centre is a display reference, not a volume centroid. All disconnected pieces of one grain move together. Coincident grain centres receive equal translations, so this view does not guarantee that every pair separates. Set `EXPLODED_GRAIN_IDS` to inspect a subset. The camera is initially fitted to the maximum slider extent so pieces remain visible while moving; zoom as desired. Shared surface vertices are duplicated only in the display.

In [61]:
from importlib import import_module, reload
_explore = reload(import_module('upxo.meshing.gbconformant.d3v2p0.exploration_views'))

EXPLODED_GRAIN_IDS = None  # None: all; or a list such as [1, 8, 15].
EXPLOSION_FACTOR = 0.25
EXPLOSION_MAX_FACTOR = 2.0
EXPLODED_CMAP = 'tab20c'
EXPLODED_OPACITY = 1.0
EXPLODED_SHOW_EDGES = False
EXPLODED_SHOW_RVE_OUTLINE = True

exploded_plotter, exploded_state, set_explosion_factor = _explore.exploded_grain_view(
    repaired_rve, grain_ids=EXPLODED_GRAIN_IDS, factor=EXPLOSION_FACTOR,
    max_factor=EXPLOSION_MAX_FACTOR, cmap=EXPLODED_CMAP, opacity=EXPLODED_OPACITY,
    show_edges=EXPLODED_SHOW_EDGES, show_outline=EXPLODED_SHOW_RVE_OUTLINE)


## Grain-adjacency graph and corresponding 3D boundaries

Each node is a grain ID; an undirected edge means the two grains share a positive-area internal triangular interface. Edge attributes contain total shared area (physical length units squared) and triangle count. RVE caps, point-only contacts and line-only contacts do not create adjacency edges. Disconnected pieces carrying the same grain ID are represented by one node. All grain IDs, including isolated nodes, remain in the returned `grain_adjacency_graph`.

By default, display a focus grain and its one-hop neighbours. `ADJACENCY_FOCUS_GRAIN = None` picks the grain with the highest displayed degree (smallest ID breaks ties). Set an exact grain ID, increase `ADJACENCY_HOPS`, or enable `ADJACENCY_FULL_GRAPH`. `ADJACENCY_MAX_NODES` prevents an unreadable, unexpectedly expensive layout; it raises instead of silently dropping nodes. The area threshold filters displayed edges only, not the returned full graph. A larger threshold can therefore change the displayed neighbourhood.

Left: graph with optional area-weighted edge widths and a red outline around the focus grain. Right: corresponding grain boundaries with matching colours. Each shared surface is drawn once, giving the focus grain colour priority, then another selected owner. Graph positions express connectivity, not physical distance; graph edges are not physical grain-boundary junction lines. The 2D graph and 3D view do not share a camera. Select another focus by changing the control and rerunning only this visualization cell.

`'spring'` layout is reproducible with a fixed seed; `'circular'` avoids iterative layout. The complete NetworkX graph is available for further neighbour/area queries, e.g. `list(grain_adjacency_graph.neighbors(8))` for an existing ID. No meshing or repairs are run.


In [59]:
from importlib import import_module, reload
_adjacency = reload(import_module('upxo.meshing.gbconformant.d3v2p0.adjacency_view'))

ADJACENCY_FOCUS_GRAIN = None
ADJACENCY_HOPS = 1
ADJACENCY_FULL_GRAPH = False
ADJACENCY_MIN_SHARED_AREA = 0.0
ADJACENCY_MAX_NODES = 150
ADJACENCY_LAYOUT = 'spring'  # Or 'circular'.
ADJACENCY_LAYOUT_SEED = 42
ADJACENCY_CMAP = 'tab20c'
ADJACENCY_NODE_SIZE = 400.0
ADJACENCY_SHOW_LABELS = True
ADJACENCY_WEIGHT_EDGE_WIDTH = True
ADJACENCY_EDGE_WIDTH = 1.5
ADJACENCY_SURFACE_OPACITY = 0.45
ADJACENCY_SURFACE_EDGES = False
ADJACENCY_WINDOW_SIZE = (1600, 800)

adjacency_plotter, grain_adjacency_graph, adjacency_display_report = _adjacency.adjacency_graph_view(
    repaired_rve, focus_grain=ADJACENCY_FOCUS_GRAIN, hops=ADJACENCY_HOPS,
    full_graph=ADJACENCY_FULL_GRAPH, minimum_area=ADJACENCY_MIN_SHARED_AREA,
    max_nodes=ADJACENCY_MAX_NODES, layout=ADJACENCY_LAYOUT, seed=ADJACENCY_LAYOUT_SEED,
    cmap=ADJACENCY_CMAP, node_size=ADJACENCY_NODE_SIZE,
    show_labels=ADJACENCY_SHOW_LABELS, weight_edge_width=ADJACENCY_WEIGHT_EDGE_WIDTH,
    edge_width=ADJACENCY_EDGE_WIDTH, surface_opacity=ADJACENCY_SURFACE_OPACITY,
    show_surface_edges=ADJACENCY_SURFACE_EDGES, window_size=ADJACENCY_WINDOW_SIZE)


{'total_grains': 373, 'total_interfaces': 1617, 'connected_components': 1, 'isolated_grain_ids': [], 'focus_grain': 295, 'displayed_grains': [8, 21, 22, 38, 41, 46, 53, 55, 58, 61, 63, 70, 71, 72, 76, 87, 89, 92, 93, 96, 105, 107, 116, 121, 127, 136, 146, 150, 155, 159, 160, 163, 173, 181, 186, 187, 192, 197, 206, 220, 229, 232, 234, 244, 251, 259, 260, 264, 269, 274, 280, 282, 283, 284, 295, 305, 311, 313, 322, 324, 327, 329, 335, 336, 340, 344, 350, 351, 352, 357, 368, 370], 'displayed_interfaces': 180, 'minimum_display_area': 0.0}
Focus neighbours (all interfaces, before display filtering):
  grain 8: shared area 0.6663888
  grain 21: shared area 0.92855431
  grain 22: shared area 18.844234
  grain 38: shared area 0.6663888
  grain 41: shared area 496.26804
  grain 46: shared area 27.041623
  grain 53: shared area 45.141883
  grain 55: shared area 0.6663888
  grain 58: shared area 3.3174648
  grain 61: shared area 0.66636114
  grain 63: shared area 0.6663888
  grain 70: shared area 